<h1 style="font-weight:900; color:#0E2841; margin:0.4em 0 0.2em;">Lerneinheit 3 · Attention</h1>

<div style="font-size:0.8em; color:#5a6472; margin-top:-4px;">Begleit-Notebook · Version 1.0</div>

<span style="color:#5a6472; font-size:0.9em;">Architektur Neuronaler Netze für Generative KI 2 · Prof. Dr. Stephan Kurpjuweit · Hochschule Worms · © 2026</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Lernziele</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Nach dieser Lerneinheit können Sie …</p>
<ul>
<li>die drei Schritte der Attention (Score, Softmax, gewichtete Summe) an einem kleinen Beispiel von Hand ausführen,</li>
<li>Query, Key und Value benennen und angeben, welche Teile der Attention Parameter haben und welche nicht,</li>
<li>die Skalierung mit 1/√dₖ begründen,</li>
<li>die Causal Mask erklären und die beiden Umsetzungen vergleichen,</li>
<li>Multi-Head Attention aufbauen und die Formen der Tensoren durch den Block verfolgen,</li>
<li>Parameterzahl und Speicherbedarf der Attention für GPT-2 small herleiten,</li>
<li>einordnen, was Attention-Gewichte über ein Modell aussagen und was nicht.</li>
</ul>

</div>
</div>

Die Folien erklären, wie ein Token seinen Kontext bekommt. In diesem Notebook steht der Code dazu: von der gewichteten Summe über Query, Key und Value bis zur Klasse `MultiHeadAttention`, die in der nächsten Lerneinheit unverändert in das GPT-Modell eingebaut wird. Der Code folgt Kapitel 3 von Sebastian Raschkas Buch *Build a Large Language Model (From Scratch)*.

Jedes Kapitel hat zwei Teile. Im ersten Teil lesen wir fertigen Code gemeinsam in der Vorlesung: An den Stellen „Kurz überlegen“ legen Sie sich zuerst auf eine Antwort fest und führen erst dann die Zelle aus. Der zweite Teil heißt **Übungen**. Die Zellen des ersten Teils laufen auch dann, wenn Sie die Übungen nicht bearbeitet haben.

**Zwei Sorten von Übungen.** Bei einem **Experiment** steht der Code fertig da. Sie notieren eine Vorhersage, führen die Zelle aus, ändern eine Bedingung und erklären das Ergebnis. Sie programmieren dabei nichts, höchstens setzen Sie eine Zahl ein. Experimente dienen dem Verständnis und machen den größten Teil der Übungen aus. Beim **Bauen** ist das Gerüst einer Klasse vorgegeben. Es fehlen die Zeilen, in denen die eigentliche Idee umgesetzt wird. Davon gibt es in dieser Lerneinheit zwei (plus eine Erweiterung nach Wahl), und zwar dort, wo Sie den Code später wieder brauchen: in Lerneinheit 4 für den Transformer-Block und im Seminarprojekt, dessen Themen zu den Interna von Sprachmodellen auf dieser Attention aufbauen. Die Prüfzellen vergleichen nicht nur einzelne Zahlen, sondern testen, ob sich Ihre Lösung richtig verhält. Im Kopf jeder Aufgabe steht, wann sie bearbeitet wird: in der Vorlesung, als Pflicht-Hausaufgabe oder nach Wahl.

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Inhalt</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<ul style="margin:0; padding-left:1.2em;"><li style="margin:5px 0;">3.1&nbsp;&nbsp;<a href="#kap-attention-ohne-parameter" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Attention ohne Parameter</a></li><li style="margin:5px 0;">3.2&nbsp;&nbsp;<a href="#kap-attention-mit-parametern" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Attention mit Parametern</a></li><li style="margin:5px 0;">3.3&nbsp;&nbsp;<a href="#kap-causal-attention" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Causal Attention</a></li><li style="margin:5px 0;">3.4&nbsp;&nbsp;<a href="#kap-multi-head-attention" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Multi-Head Attention</a></li><li style="margin:5px 0;">3.5&nbsp;&nbsp;<a href="#kap-attention-in-gro-en-llms" style="color:#1a1a1a; text-decoration:none; border-bottom:1px solid #E8B029;">Attention in großen LLMs</a></li></ul>

</div>
</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E75B6; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Bevor es losgeht</div>
<div style="background:#EAF2FA; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>In <strong>Colab</strong> klicken Sie zuerst auf „Kopie in Drive speichern“. Ohne diesen Klick gehen Ihre Änderungen beim Schließen verloren.</p>
<p><strong>Lokal</strong> starten Sie JupyterLab über das Startskript im Kursordner (<code>start_lokal</code>). Die README im Kursordner beschreibt beide Wege.</p>
<p>Die folgende Zelle prüft Ihre Umgebung. In Colab installiert sie dabei das fehlende Paket <code>tiktoken</code>. Eine GPU brauchen Sie für diese Lerneinheit nicht.</p>

</div>
</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: Umgebung prüfen — einfach ausführen</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">importlib</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">subprocess</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">sys</span>&#10;<span style="color: #008000; font-weight: bold">from</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">importlib.metadata</span><span style="color: #BBB"> </span><span style="color: #008000; font-weight: bold">import</span> version</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; In Colab ist das Modul <code>google.colab</code> schon geladen. Daran erkennt die Zelle, wo sie läuft.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>IN_COLAB <span style="color: #666">=</span> <span style="color: #BA2121">"google.colab"</span> <span style="color: #A2F; font-weight: bold">in</span> sys<span style="color: #666">.</span>modules&#10;v <span style="color: #666">=</span> sys<span style="color: #666">.</span>version_info&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span><span style="color: #BA2121">'✓'</span><span style="color: #BBB"> </span><span style="color: #008000; font-weight: bold">if</span><span style="color: #BBB"> </span>(v<span style="color: #666">.</span>major,<span style="color: #BBB"> </span>v<span style="color: #666">.</span>minor)<span style="color: #BBB"> </span><span style="color: #666">&gt;=</span><span style="color: #BBB"> </span>(<span style="color: #666">3</span>,<span style="color: #BBB"> </span><span style="color: #666">10</span>)<span style="color: #BBB"> </span><span style="color: #008000; font-weight: bold">else</span><span style="color: #BBB"> </span><span style="color: #BA2121">'✗'</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> Python    </span><span style="color: #A45A77; font-weight: bold">{</span>v<span style="color: #666">.</span>major<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">.</span><span style="color: #A45A77; font-weight: bold">{</span>v<span style="color: #666">.</span>minor<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">.</span><span style="color: #A45A77; font-weight: bold">{</span>v<span style="color: #666">.</span>micro<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Die Zelle versucht, jedes Paket zu importieren. Fehlt eines, installiert sie es in Colab nach. Lokal verweist sie auf das Startskript.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">for</span> paket <span style="color: #A2F; font-weight: bold">in</span> [<span style="color: #BA2121">"torch"</span>, <span style="color: #BA2121">"tiktoken"</span>, <span style="color: #BA2121">"matplotlib"</span>]:&#10;    <span style="color: #008000; font-weight: bold">try</span>:&#10;        importlib<span style="color: #666">.</span>import_module(paket)&#10;    <span style="color: #008000; font-weight: bold">except</span> <span style="color: #CB3F38; font-weight: bold">ModuleNotFoundError</span>:&#10;        <span style="color: #008000; font-weight: bold">if</span> <span style="color: #A2F; font-weight: bold">not</span> IN_COLAB:&#10;            <span style="color: #008000">print</span>(<span style="color: #BA2121">f"✗ </span><span style="color: #A45A77; font-weight: bold">{</span>paket<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">10s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> fehlt. Bitte JupyterLab über das Startskript starten (siehe README)."</span>)&#10;            <span style="color: #008000; font-weight: bold">continue</span>&#10;        subprocess<span style="color: #666">.</span>run([sys<span style="color: #666">.</span>executable, <span style="color: #BA2121">"-m"</span>, <span style="color: #BA2121">"pip"</span>, <span style="color: #BA2121">"install"</span>, <span style="color: #BA2121">"-q"</span>, paket], check<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>)&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"✓ </span><span style="color: #A45A77; font-weight: bold">{</span>paket<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">10s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> </span><span style="color: #A45A77; font-weight: bold">{</span>version(paket)<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Zum Schluss zeigt die Zelle das Rechengerät an. Für diese Lerneinheit reicht die CPU.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">if</span> <span style="color: #BA2121">"torch"</span> <span style="color: #A2F; font-weight: bold">in</span> sys<span style="color: #666">.</span>modules:&#10;    <span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">torch</span>&#10;    <span style="color: #008000; font-weight: bold">if</span> torch<span style="color: #666">.</span>cuda<span style="color: #666">.</span>is_available():&#10;        geraet <span style="color: #666">=</span> <span style="color: #BA2121">f"CUDA-GPU (</span><span style="color: #A45A77; font-weight: bold">{</span>torch<span style="color: #666">.</span>cuda<span style="color: #666">.</span>get_device_name(<span style="color: #666">0</span>)<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">)"</span>&#10;    <span style="color: #008000; font-weight: bold">elif</span> torch<span style="color: #666">.</span>backends<span style="color: #666">.</span>mps<span style="color: #666">.</span>is_available():&#10;        geraet <span style="color: #666">=</span> <span style="color: #BA2121">"Apple-GPU (MPS)"</span>&#10;    <span style="color: #008000; font-weight: bold">else</span>:&#10;        geraet <span style="color: #666">=</span> <span style="color: #BA2121">"CPU"</span>&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"✓ Gerät      </span><span style="color: #A45A77; font-weight: bold">{</span>geraet<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)</code></pre></details>

In [ ]:
# Vorgabe: Umgebung prüfen — einfach ausführen
import importlib
import subprocess
import sys
from importlib.metadata import version

IN_COLAB = "google.colab" in sys.modules
v = sys.version_info
print(f"{'✓' if (v.major, v.minor) >= (3, 10) else '✗'} Python    {v.major}.{v.minor}.{v.micro}")

for paket in ["torch", "tiktoken", "matplotlib"]:
    try:
        importlib.import_module(paket)
    except ModuleNotFoundError:
        if not IN_COLAB:
            print(f"✗ {paket:10s} fehlt. Bitte JupyterLab über das Startskript starten (siehe README).")
            continue
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", paket], check=True)
    print(f"✓ {paket:10s} {version(paket)}")

if "torch" in sys.modules:
    import torch
    if torch.cuda.is_available():
        geraet = f"CUDA-GPU ({torch.cuda.get_device_name(0)})"
    elif torch.backends.mps.is_available():
        geraet = "Apple-GPU (MPS)"
    else:
        geraet = "CPU"
    print(f"✓ Gerät      {geraet}")

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: Pakete und Rechenbeispiel — einfach ausführen</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">math</span>&#10;&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">torch</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">torch.nn</span><span style="color: #BBB"> </span><span style="color: #008000; font-weight: bold">as</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">nn</span>&#10;&#10;torch<span style="color: #666">.</span>set_printoptions(precision<span style="color: #666">=4</span>, sci_mode<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Das Rechenbeispiel aus den Folien: sechs Wort-Tokens mit je drei Dimensionen. Die Werte sind frei gewählt. In GPT-2 würde der Satz in 22 BPE-Tokens mit je 768 Dimensionen zerlegt.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>woerter <span style="color: #666">=</span> [<span style="color: #BA2121">"Übermorgen"</span>, <span style="color: #BA2121">"fährt"</span>, <span style="color: #BA2121">"das"</span>, <span style="color: #BA2121">"Rheinschiff"</span>, <span style="color: #BA2121">"nach"</span>, <span style="color: #BA2121">"Worms"</span>]&#10;inputs <span style="color: #666">=</span> torch<span style="color: #666">.</span>tensor(&#10;    [[<span style="color: #666">0.1</span>, <span style="color: #666">0.1</span>, <span style="color: #666">0.9</span>],   <span style="color: #3D7B7B; font-style: italic"># Übermorgen</span>&#10;     [<span style="color: #666">0.9</span>, <span style="color: #666">0.3</span>, <span style="color: #666">0.2</span>],   <span style="color: #3D7B7B; font-style: italic"># fährt</span>&#10;     [<span style="color: #666">0.1</span>, <span style="color: #666">0.1</span>, <span style="color: #666">0.1</span>],   <span style="color: #3D7B7B; font-style: italic"># das</span>&#10;     [<span style="color: #666">0.8</span>, <span style="color: #666">0.6</span>, <span style="color: #666">0.1</span>],   <span style="color: #3D7B7B; font-style: italic"># Rheinschiff</span>&#10;     [<span style="color: #666">0.3</span>, <span style="color: #666">0.7</span>, <span style="color: #666">0.1</span>],   <span style="color: #3D7B7B; font-style: italic"># nach</span>&#10;     [<span style="color: #666">0.2</span>, <span style="color: #666">0.9</span>, <span style="color: #666">0.1</span>]])  <span style="color: #3D7B7B; font-style: italic"># Worms</span></code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Zum Vergleich mit dem Buch: Raschkas Beispiel „Your journey starts with one step“ mit seinen Werten. Wenn Sie einzelne Zellen mit dem Buch abgleichen möchten, ersetzen Sie dort <code>inputs</code> durch <code>inputs_buch</code>. Beschriftungen, Query-Index (hier 3, im Buch zunächst 1) und Prüfzellen gelten aber weiterhin für das deutsche Beispiel.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>woerter_buch <span style="color: #666">=</span> [<span style="color: #BA2121">"Your"</span>, <span style="color: #BA2121">"journey"</span>, <span style="color: #BA2121">"starts"</span>, <span style="color: #BA2121">"with"</span>, <span style="color: #BA2121">"one"</span>, <span style="color: #BA2121">"step"</span>]&#10;inputs_buch <span style="color: #666">=</span> torch<span style="color: #666">.</span>tensor(&#10;    [[<span style="color: #666">0.43</span>, <span style="color: #666">0.15</span>, <span style="color: #666">0.89</span>], [<span style="color: #666">0.55</span>, <span style="color: #666">0.87</span>, <span style="color: #666">0.66</span>], [<span style="color: #666">0.57</span>, <span style="color: #666">0.85</span>, <span style="color: #666">0.64</span>],&#10;     [<span style="color: #666">0.22</span>, <span style="color: #666">0.58</span>, <span style="color: #666">0.33</span>], [<span style="color: #666">0.77</span>, <span style="color: #666">0.25</span>, <span style="color: #666">0.10</span>], [<span style="color: #666">0.05</span>, <span style="color: #666">0.80</span>, <span style="color: #666">0.55</span>]])</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; <code>zeige</code> druckt eine Matrix mit Zeilen- und Spaltenbeschriftung, damit man Gewichte lesen kann, ohne Indizes zu zählen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">zeige</span>(matrix, zeilen<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">None</span>, spalten<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">None</span>, stellen<span style="color: #666">=2</span>):&#10;    zeilen <span style="color: #666">=</span> zeilen <span style="color: #A2F; font-weight: bold">or</span> [<span style="color: #BA2121">""</span>] <span style="color: #666">*</span> matrix<span style="color: #666">.</span>shape[<span style="color: #666">0</span>]&#10;    spalten <span style="color: #666">=</span> spalten <span style="color: #A2F; font-weight: bold">or</span> [<span style="color: #BA2121">""</span>] <span style="color: #666">*</span> matrix<span style="color: #666">.</span>shape[<span style="color: #666">1</span>]&#10;    breite <span style="color: #666">=</span> <span style="color: #008000">max</span>(<span style="color: #008000">len</span>(w) <span style="color: #008000; font-weight: bold">for</span> w <span style="color: #A2F; font-weight: bold">in</span> zeilen <span style="color: #666">+</span> spalten) <span style="color: #666">+</span> <span style="color: #666">1</span>&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">" "</span> <span style="color: #666">*</span> breite <span style="color: #666">+</span> <span style="color: #BA2121">""</span><span style="color: #666">.</span>join(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span>s<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;</span><span style="color: #A45A77; font-weight: bold">{</span>breite<span style="color: #A45A77; font-weight: bold">}}</span><span style="color: #BA2121">"</span> <span style="color: #008000; font-weight: bold">for</span> s <span style="color: #A2F; font-weight: bold">in</span> spalten))&#10;    <span style="color: #008000; font-weight: bold">for</span> name, zeile <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">zip</span>(zeilen, matrix):&#10;        <span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span>name<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;</span><span style="color: #A45A77; font-weight: bold">{</span>breite<span style="color: #A45A77; font-weight: bold">}}</span><span style="color: #BA2121">"</span> <span style="color: #666">+</span> <span style="color: #BA2121">""</span><span style="color: #666">.</span>join(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span><span style="color: #008000">float</span>(x)<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;</span><span style="color: #A45A77; font-weight: bold">{</span>breite<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">.</span><span style="color: #A45A77; font-weight: bold">{</span>stellen<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span> <span style="color: #008000; font-weight: bold">for</span> x <span style="color: #A2F; font-weight: bold">in</span> zeile))&#10;&#10;&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"inputs hat die Form"</span>, <span style="color: #008000">tuple</span>(inputs<span style="color: #666">.</span>shape), <span style="color: #BA2121">"(T Tokens × d Dimensionen)"</span>)&#10;zeige(inputs, woerter, [<span style="color: #BA2121">"d1"</span>, <span style="color: #BA2121">"d2"</span>, <span style="color: #BA2121">"d3"</span>], stellen<span style="color: #666">=1</span>)</code></pre></details>

In [ ]:
# Vorgabe: Pakete und Rechenbeispiel — einfach ausführen
import math

import torch
import torch.nn as nn

torch.set_printoptions(precision=4, sci_mode=False)

woerter = ["Übermorgen", "fährt", "das", "Rheinschiff", "nach", "Worms"]
inputs = torch.tensor(
    [[0.1, 0.1, 0.9],   # Übermorgen
     [0.9, 0.3, 0.2],   # fährt
     [0.1, 0.1, 0.1],   # das
     [0.8, 0.6, 0.1],   # Rheinschiff
     [0.3, 0.7, 0.1],   # nach
     [0.2, 0.9, 0.1]])  # Worms

woerter_buch = ["Your", "journey", "starts", "with", "one", "step"]
inputs_buch = torch.tensor(
    [[0.43, 0.15, 0.89], [0.55, 0.87, 0.66], [0.57, 0.85, 0.64],
     [0.22, 0.58, 0.33], [0.77, 0.25, 0.10], [0.05, 0.80, 0.55]])


def zeige(matrix, zeilen=None, spalten=None, stellen=2):
    zeilen = zeilen or [""] * matrix.shape[0]
    spalten = spalten or [""] * matrix.shape[1]
    breite = max(len(w) for w in zeilen + spalten) + 1
    print(" " * breite + "".join(f"{s:>{breite}}" for s in spalten))
    for name, zeile in zip(zeilen, matrix):
        print(f"{name:>{breite}}" + "".join(f"{float(x):>{breite}.{stellen}f}" for x in zeile))


print("inputs hat die Form", tuple(inputs.shape), "(T Tokens × d Dimensionen)")
zeige(inputs, woerter, ["d1", "d2", "d3"], stellen=1)

Das Rechenbeispiel verwendet wieder den Satz aus Lerneinheit 2. Diesmal ist er in sechs Wort-Tokens zerlegt, damit die ganze Rechnung auf eine Folie passt. Alle Zahlen auf den Folien stammen aus diesem Tensor.

## <a id="kap-attention-ohne-parameter"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;">3.1&nbsp;&nbsp;Attention ohne Parameter</span>

Attention beantwortet für jedes Token die Frage, wie viel es von jedem anderen Token in seine neue Darstellung aufnimmt. Die Antwort entsteht in drei Schritten: Ähnlichkeit messen, in Gewichte verwandeln, gewichtet summieren. In diesem Kapitel kommt die Attention noch ohne gelernte Parameter aus. Alle Werte werden direkt aus den Eingabevektoren berechnet (Raschka, Abschnitt 3.3).

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Die Query ist das Token, für das wir den Kontext berechnen: „Rheinschiff“, Index 3.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>query <span style="color: #666">=</span> inputs[<span style="color: #666">3</span>]</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Wir berechnen das Skalarprodukt der Query mit jedem der sechs Eingabevektoren, also auch mit ihr selbst. <code>inputs @ query</code> berechnet alle sechs auf einmal.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>scores <span style="color: #666">=</span> inputs <span style="color: #666">@</span> query&#10;&#10;<span style="color: #008000; font-weight: bold">for</span> wort, s <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">zip</span>(woerter, scores):&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span>wort<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;12s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">  </span><span style="color: #A45A77; font-weight: bold">{</span>s<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)</code></pre></details>

In [ ]:
# Vorgabe: Schritt 1, Ähnlichkeit messen — einfach ausführen
query = inputs[3]

scores = inputs @ query

for wort, s in zip(woerter, scores):
    print(f"{wort:>12s}  {s:.2f}")

Das Skalarprodukt ist groß, wenn zwei Vektoren in dieselbe Richtung zeigen und lang sind: xᵀy = ‖x‖ ‖y‖ cos θ. Anders als bei der Kosinus-Ähnlichkeit wird hier nicht durch die Vektorlängen geteilt, die Länge geht also in den Score ein. Deshalb bekommt der kurze Vektor von „das“ überall kleine Scores.

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Der naheliegende Weg wäre, jeden Score durch die Summe aller Scores zu teilen. Das ergibt Gewichte mit Summe 1, funktioniert aber nur, solange alle Scores positiv sind.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>naiv <span style="color: #666">=</span> scores <span style="color: #666">/</span> scores<span style="color: #666">.</span>sum()</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Die Softmax wendet zuerst <code>exp</code> auf jeden Score an und teilt dann durch die Summe. Dadurch sind alle Gewichte positiv und summieren sich zu 1, und große Scores werden stärker betont. <code>dim=0</code> bedeutet, dass über die sechs Tokens normiert wird.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>gewichte <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(scores, dim<span style="color: #666">=0</span>)&#10;&#10;<span style="color: #008000; font-weight: bold">for</span> wort, n, g <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">zip</span>(woerter, naiv, gewichte):&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span>wort<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;12s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">  naiv </span><span style="color: #A45A77; font-weight: bold">{</span>n<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">   softmax </span><span style="color: #A45A77; font-weight: bold">{</span>g<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span><span style="color: #BA2121">'Summe'</span><span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;12s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">  naiv </span><span style="color: #A45A77; font-weight: bold">{</span>naiv<span style="color: #666">.</span>sum()<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">   softmax </span><span style="color: #A45A77; font-weight: bold">{</span>gewichte<span style="color: #666">.</span>sum()<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)</code></pre></details>

In [ ]:
# Vorgabe: Schritt 2, Softmax — einfach ausführen
naiv = scores / scores.sum()

gewichte = torch.softmax(scores, dim=0)

for wort, n, g in zip(woerter, naiv, gewichte):
    print(f"{wort:>12s}  naiv {n:.2f}   softmax {g:.2f}")
print(f"{'Summe':>12s}  naiv {naiv.sum():.2f}   softmax {gewichte.sum():.2f}")

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Der Kontextvektor ist die Summe der sechs Eingabevektoren, jeder mit seinem Gewicht multipliziert. <code>gewichte @ inputs</code> berechnet diese Summe.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>kontext <span style="color: #666">=</span> gewichte <span style="color: #666">@</span> inputs&#10;&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Kontextvektor für „Rheinschiff“:"</span>, kontext)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Zur Kontrolle berechnen wir dieselbe Summe noch einmal mit einer Schleife. Beide Wege liefern dasselbe.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>summe <span style="color: #666">=</span> torch<span style="color: #666">.</span>zeros(<span style="color: #666">3</span>)&#10;<span style="color: #008000; font-weight: bold">for</span> g, x <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">zip</span>(gewichte, inputs):&#10;    summe <span style="color: #666">+=</span> g <span style="color: #666">*</span> x&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"als Schleife:                  "</span>, summe)</code></pre></details>

In [ ]:
# Vorgabe: Schritt 3, gewichtete Summe — einfach ausführen
kontext = gewichte @ inputs

print("Kontextvektor für „Rheinschiff“:", kontext)
summe = torch.zeros(3)
for g, x in zip(gewichte, inputs):
    summe += g * x
print("als Schleife:                  ", summe)

Der Kontextvektor ist die kontextuelle Repräsentation aus Lerneinheit 2: ein Vektor für „Rheinschiff“, in dem Information aus dem ganzen Satz steckt. Bis hierher haben wir für ein Token gerechnet. Um alle sechs Kontextvektoren auf einmal zu berechnen, ersetzen wir die einzelne Query durch die ganze Eingabematrix: Aus dem Score-Vektor wird eine 6 × 6-Matrix, die Softmax läuft je Zeile, und die gewichtete Summe wird zum Matrixprodukt.

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Die nächste Zelle berechnet die Gewichte für alle sechs Tokens auf einmal, eine Zeile je Query. Legen Sie sich fest, bevor Sie sie ausführen: <strong>Auf welches Token legt „nach“ das höchste Gewicht?</strong></p>

</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; <code>inputs @ inputs.T</code> liefert alle 36 Skalarprodukte als Matrix [6, 6]. Zeile i enthält die Scores der Query i gegen alle sechs Keys.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>alle_scores <span style="color: #666">=</span> inputs <span style="color: #666">@</span> inputs<span style="color: #666">.</span>T</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; <code>dim=-1</code> normiert jede Zeile für sich. Jede Zeile ist danach eine Verteilung mit Summe 1.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>alle_gewichte <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(alle_scores, dim<span style="color: #666">=-1</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Eine weitere Matrixmultiplikation liefert alle sechs Kontextvektoren: [6, 6] @ [6, 3] = [6, 3].</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>alle_kontexte <span style="color: #666">=</span> alle_gewichte <span style="color: #666">@</span> inputs&#10;&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Gewichte, Zeile = Query, Spalte = Key"</span>)&#10;zeige(alle_gewichte, woerter, woerter)&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000; font-weight: bold">for</span> i, wort <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">enumerate</span>(woerter):&#10;    j <span style="color: #666">=</span> <span style="color: #008000">int</span>(alle_gewichte[i]<span style="color: #666">.</span>argmax())&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span>wort<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;12s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> schaut am stärksten auf </span><span style="color: #A45A77; font-weight: bold">{</span>woerter[j]<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">&gt;12s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> (</span><span style="color: #A45A77; font-weight: bold">{</span>alle_gewichte[i,<span style="color: #BBB"> </span>j]<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">)"</span>)&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Form der Kontextvektoren:"</span>, <span style="color: #008000">tuple</span>(alle_kontexte<span style="color: #666">.</span>shape))</code></pre></details>

In [ ]:
# Vorgabe: alle Gewichte auf einmal — einfach ausführen
alle_scores = inputs @ inputs.T

alle_gewichte = torch.softmax(alle_scores, dim=-1)

alle_kontexte = alle_gewichte @ inputs

print("Gewichte, Zeile = Query, Spalte = Key")
zeige(alle_gewichte, woerter, woerter)
print()
for i, wort in enumerate(woerter):
    j = int(alle_gewichte[i].argmax())
    print(f"{wort:>12s} schaut am stärksten auf {woerter[j]:>12s} ({alle_gewichte[i, j]:.2f})")
print()
print("Form der Kontextvektoren:", tuple(alle_kontexte.shape))

Das Token „nach“ gibt sein höchstes Gewicht an „Worms“, ein Token, das im Satz erst später kommt. Bei einem Sprachmodell, das das nächste Token vorhersagen soll, darf das nicht passieren. Darum kümmert sich Kapitel 3.3. Vorher klären wir, was an dieser Rechnung noch fehlt: Ohne Parameter hängt das Gewicht nur davon ab, wie ähnlich sich die Embeddings sind. Außerdem hat jedes Token nur eine Rolle: Derselbe Vektor dient zum Vergleich mit den anderen Tokens und geht auch in die gewichtete Summe ein.

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">3.1.1&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">8 min</span>🔬 Aufgabe 3.1 #1 · Experiment · Hausaufgabe, Pflicht</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie führen die drei Schritte selbst aus und sehen, wo der Kontextvektor im Vergleich zu den sechs Eingabevektoren liegt.</em></p>
<p>Rechnen Sie die drei Schritte für die Query <strong>„Worms“</strong> (Index 5) von Hand nach, mit Taschenrechner auf zwei Stellen. Tragen Sie Ihre Werte in die Zelle ein. Die Prüfzelle vergleicht sie mit der Rechnung in PyTorch, die darunter schon fertig steht.</p>
<p>Schreiben Sie vorab Ihre Vorhersage in die Variable <code>vorhersage</code>: Welches Token bekommt das höchste Gewicht?</p>
<p>Zusatzfrage, falls Sie früher fertig sind: In welcher Komponente liegt der Kontextvektor am nächsten am Vektor von „Worms“, und warum?</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Für die Handrechnung reicht e^x auf dem Taschenrechner: exp(0,86) ≈ 2,36. Die Gewichte summieren sich zu 1, und der Kontextvektor ist die gewichtete Summe der sechs Zeilen von <code>inputs</code>.</p></div></details>

</div>
</div>

In [ ]:
vorhersage = "..."                      # Wort aus `woerter`

# Ihre Handrechnung, zwei Stellen
hand_scores = [0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
hand_gewichte = [0.00, 0.00, 0.00, 0.00, 0.00, 0.00]
hand_kontext = [0.00, 0.00, 0.00]

# Zur Kontrolle: dieselbe Rechnung in PyTorch
scores_worms = inputs @ inputs[5]
gewichte_worms = torch.softmax(scores_worms, dim=0)
kontext_worms = gewichte_worms @ inputs
for wort, s, g in zip(woerter, scores_worms, gewichte_worms):
    print(f"{wort:>12s}  Score {s:.2f}  Gewicht {g:.2f}")
print("Kontextvektor:", kontext_worms)

In [ ]:
# Prüfzelle — einfach ausführen
assert vorhersage in woerter, "Tragen Sie ein Wort aus `woerter` in `vorhersage` ein"
assert torch.allclose(torch.tensor(hand_scores), scores_worms, atol=0.015), "Scores: Skalarprodukt der Query Worms mit jeder Zeile von inputs, auf zwei Stellen"
assert torch.allclose(torch.tensor(hand_gewichte), gewichte_worms, atol=0.015), "Gewichte: exp(Score) geteilt durch die Summe aller exp(Scores)"
assert torch.allclose(torch.tensor(hand_kontext), kontext_worms, atol=0.03), "Kontextvektor: Summe aus Gewicht mal Zeile über alle sechs Tokens"
print("Höchstes Gewicht hat:", woerter[int(gewichte_worms.argmax())], "— Ihre Vorhersage:", vorhersage)
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>🔬 Aufgabe 3.1 #2 · Experiment · Hausaufgabe, nach Wahl</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie untersuchen, wie sich die Gewichtsverteilung ändert, wenn nur die Query länger wird.</em></p>
<p>Die Zelle zeichnet die Gewichtsmatrix als Heatmap und berechnet dann die Zeile für „das“ noch einmal. Dabei wird die Query mit <code>faktor</code> multipliziert, die Keys bleiben unverändert. Setzen Sie <code>faktor</code> auf 10 und schreiben Sie vorher Ihre Vorhersage auf: Wird die Zeile flacher, schärfer, oder bleibt sie gleich? Probieren Sie danach <code>faktor = 0.1</code> aus.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Die Richtung der Query bestimmt, wohin die Spitzen zeigen. Beobachten Sie, was ihre Länge mit der Höhe der Spitzen macht.</p></div></details>

</div>
</div>

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(4.5, 4))
bild = ax.imshow(alle_gewichte, cmap="Blues", vmin=0, vmax=0.4)
ax.set_xticks(range(6), woerter, rotation=60, ha="right")
ax.set_yticks(range(6), woerter)
ax.set_xlabel("Key"); ax.set_ylabel("Query")
fig.colorbar(bild, ax=ax, shrink=0.8)
plt.show()

vorhersage = "..."           # "flacher", "schärfer" oder "gleich"
faktor = 1                   # hier ändern: 10, dann 0.1

gewichte_das = torch.softmax(faktor * inputs[2] @ inputs.T, dim=0)
print(f"Zeile das, Faktor {faktor}:", gewichte_das)

In [ ]:
# Prüfzelle — einfach ausführen
assert vorhersage in ("flacher", "schärfer", "gleich"), "Tragen Sie flacher, schärfer oder gleich ein"
assert faktor != 1, "Setzen Sie faktor auf 10 (und danach auf 0.1)"
assert torch.allclose(gewichte_das, torch.softmax(faktor * inputs[2] @ inputs.T, dim=0), atol=1e-5)
_spanne = float(gewichte_das.max() - gewichte_das.min())
print(f"Spanne der Gewichte bei Faktor {faktor}: {_spanne:.3f} (bei Faktor 1: {float(alle_gewichte[2].max() - alle_gewichte[2].min()):.3f}) — Ihre Vorhersage: {vorhersage}")
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 3.1 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Warum nimmt man die Softmax und teilt die Scores nicht einfach durch ihre Summe?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Die Zeile für „das“ ist fast gleichmäßig. Was ist der Grund, und was ändert sich, wenn die Keys sehr lang wären?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Was wäre anders, wenn man statt <code>dim=-1</code> über <code>dim=0</code> normieren würde?</p>

</div>

## <a id="kap-attention-mit-parametern"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;">3.2&nbsp;&nbsp;Attention mit Parametern</span>

Bis jetzt hat jedes Token nur eine Rolle. Drei gelernte Matrizen trennen die Rollen: Die **Query** ist das, womit ein Token sucht, der **Key** das, womit es gefunden wird, der **Value** das, was es in die Summe einbringt (Raschka, Abschnitt 3.4). Die drei Schritte bleiben gleich. Neu ist nur, dass wir Queries mit Keys vergleichen statt Eingabevektoren miteinander, und dass wir Values statt Eingaben aufsummieren.

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: Query, Key, Value für ein Token — einfach ausführen</span>&#10;d_in <span style="color: #666">=</span> inputs<span style="color: #666">.</span>shape[<span style="color: #666">1</span>]   <span style="color: #3D7B7B; font-style: italic"># 3</span>&#10;d_out <span style="color: #666">=</span> <span style="color: #666">2</span>                <span style="color: #3D7B7B; font-style: italic"># klein, damit man die Zahlen lesen kann</span></code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Drei Matrizen der Form [d_in, d_out] mit zufälligen Startwerten. <code>requires_grad=False</code> setzen wir nur hier, weil wir noch nicht trainieren.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">123</span>)&#10;W_query <span style="color: #666">=</span> nn<span style="color: #666">.</span>Parameter(torch<span style="color: #666">.</span>rand(d_in, d_out), requires_grad<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>)&#10;W_key <span style="color: #666">=</span> nn<span style="color: #666">.</span>Parameter(torch<span style="color: #666">.</span>rand(d_in, d_out), requires_grad<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>)&#10;W_value <span style="color: #666">=</span> nn<span style="color: #666">.</span>Parameter(torch<span style="color: #666">.</span>rand(d_in, d_out), requires_grad<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Die Query berechnen wir nur für „Rheinschiff“, Keys und Values für alle sechs Tokens.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>x_3 <span style="color: #666">=</span> inputs[<span style="color: #666">3</span>]&#10;query_3 <span style="color: #666">=</span> x_3 <span style="color: #666">@</span> W_query&#10;keys <span style="color: #666">=</span> inputs <span style="color: #666">@</span> W_key&#10;values <span style="color: #666">=</span> inputs <span style="color: #666">@</span> W_value&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"query_3:"</span>, query_3, <span style="color: #BA2121">" keys:"</span>, <span style="color: #008000">tuple</span>(keys<span style="color: #666">.</span>shape), <span style="color: #BA2121">" values:"</span>, <span style="color: #008000">tuple</span>(values<span style="color: #666">.</span>shape))</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Schritt 1 und 2 laufen wie vorher. Neu sind der Vergleich der Query mit den Keys und die Skalierung mit √d_k, die wir gleich begründen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>scores_3 <span style="color: #666">=</span> query_3 <span style="color: #666">@</span> keys<span style="color: #666">.</span>T&#10;d_k <span style="color: #666">=</span> keys<span style="color: #666">.</span>shape[<span style="color: #666">-1</span>]&#10;gewichte_3 <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(scores_3 <span style="color: #666">/</span> d_k<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Gewichte:"</span>, gewichte_3)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>④</b>&nbsp; Schritt 3 bildet die gewichtete Summe der Values, nicht der Eingaben.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>kontext_3 <span style="color: #666">=</span> gewichte_3 <span style="color: #666">@</span> values&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Kontextvektor:"</span>, kontext_3, <span style="color: #BA2121">"mit Form"</span>, <span style="color: #008000">tuple</span>(kontext_3<span style="color: #666">.</span>shape))</code></pre></details>

In [ ]:
# Vorgabe: Query, Key, Value für ein Token — einfach ausführen
d_in = inputs.shape[1]   # 3
d_out = 2                # klein, damit man die Zahlen lesen kann

torch.manual_seed(123)
W_query = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_key = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
W_value = nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)

x_3 = inputs[3]
query_3 = x_3 @ W_query
keys = inputs @ W_key
values = inputs @ W_value
print("query_3:", query_3, " keys:", tuple(keys.shape), " values:", tuple(values.shape))

scores_3 = query_3 @ keys.T
d_k = keys.shape[-1]
gewichte_3 = torch.softmax(scores_3 / d_k**0.5, dim=-1)
print("Gewichte:", gewichte_3)

kontext_3 = gewichte_3 @ values
print("Kontextvektor:", kontext_3, "mit Form", tuple(kontext_3.shape))

Der Kontextvektor hat jetzt die Breite `d_out` statt `d_in`. In GPT-2 sind beide 768. Neu ist die Division durch √dₖ. Im Beispiel mit dₖ = 2 spielt sie kaum eine Rolle. Bei dₖ = 64 oder 768 entscheidet sie mit darüber, wie gut das Training anläuft.

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Die nächste Zelle nimmt vier Scores [2, 1, 0.5, −1] und berechnet die Softmax. Dann multipliziert sie alle Scores mit 8. Legen Sie sich fest: <strong>Was passiert mit den Gewichten, wenn die Scores mit 8 multipliziert werden?</strong></p>

</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: warum durch √d_k geteilt wird — einfach ausführen</span>&#10;torch<span style="color: #666">.</span>set_printoptions(precision<span style="color: #666">=4</span>, sci_mode<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>)&#10;s <span style="color: #666">=</span> torch<span style="color: #666">.</span>tensor([<span style="color: #666">2.0</span>, <span style="color: #666">1.0</span>, <span style="color: #666">0.5</span>, <span style="color: #666">-1.0</span>])&#10;p <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(s, dim<span style="color: #666">=0</span>)&#10;p8 <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(<span style="color: #666">8</span> <span style="color: #666">*</span> s, dim<span style="color: #666">=0</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Softmax der Scores:    "</span>, p)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Softmax der Scores × 8:"</span>, p8)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Die Ableitung des größten Softmax-Ausgangs nach seinem eigenen Score ist p(1 − p). Ist die Verteilung fast One-Hot, liegt p nahe 1 und die Ableitung nahe null. Entsprechend klein ist dann der Gradient, der durch die Softmax zu den Scores und weiter zu W_query und W_key fließt.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000">print</span>(<span style="color: #BA2121">f"p(1 − p) für das größte Gewicht: </span><span style="color: #A45A77; font-weight: bold">{</span>p[<span style="color: #666">0</span>]<span style="color: #BBB"> </span><span style="color: #666">*</span><span style="color: #BBB"> </span>(<span style="color: #666">1</span><span style="color: #BBB"> </span><span style="color: #666">-</span><span style="color: #BBB"> </span>p[<span style="color: #666">0</span>])<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.4f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> ohne Faktor, </span><span style="color: #A45A77; font-weight: bold">{</span>p8[<span style="color: #666">0</span>]<span style="color: #BBB"> </span><span style="color: #666">*</span><span style="color: #BBB"> </span>(<span style="color: #666">1</span><span style="color: #BBB"> </span><span style="color: #666">-</span><span style="color: #BBB"> </span>p8[<span style="color: #666">0</span>])<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2e</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> mit Faktor 8"</span>)&#10;torch<span style="color: #666">.</span>set_printoptions(precision<span style="color: #666">=4</span>, sci_mode<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Woher kommen große Scores? Haben q und k unabhängige Einträge mit Mittelwert 0 und Varianz σ², hat ihr Skalarprodukt die Varianz d_k · σ⁴, bei σ = 1 also d_k (Vaswani et al. 2017, Fußnote 4). Die Messung bestätigt das für σ = 1.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">0</span>)&#10;<span style="color: #008000; font-weight: bold">for</span> d <span style="color: #A2F; font-weight: bold">in</span> (<span style="color: #666">3</span>, <span style="color: #666">64</span>, <span style="color: #666">768</span>):&#10;    q <span style="color: #666">=</span> torch<span style="color: #666">.</span>randn(<span style="color: #666">10_000</span>, d)&#10;    k <span style="color: #666">=</span> torch<span style="color: #666">.</span>randn(<span style="color: #666">10_000</span>, d)&#10;    skalar <span style="color: #666">=</span> (q <span style="color: #666">*</span> k)<span style="color: #666">.</span>sum(dim<span style="color: #666">=1</span>)&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"d = </span><span style="color: #A45A77; font-weight: bold">{</span>d<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">3d</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">: Varianz der Skalarprodukte </span><span style="color: #A45A77; font-weight: bold">{</span>skalar<span style="color: #666">.</span>var()<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">7.1f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">, nach Division durch √d </span><span style="color: #A45A77; font-weight: bold">{</span>(skalar<span style="color: #BBB"> </span><span style="color: #666">/</span><span style="color: #BBB"> </span>d<span style="color: #666">**0.5</span>)<span style="color: #666">.</span>var()<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)</code></pre></details>

In [ ]:
# Vorgabe: warum durch √d_k geteilt wird — einfach ausführen
torch.set_printoptions(precision=4, sci_mode=True)
s = torch.tensor([2.0, 1.0, 0.5, -1.0])
p = torch.softmax(s, dim=0)
p8 = torch.softmax(8 * s, dim=0)
print("Softmax der Scores:    ", p)
print("Softmax der Scores × 8:", p8)
print(f"p(1 − p) für das größte Gewicht: {p[0] * (1 - p[0]):.4f} ohne Faktor, {p8[0] * (1 - p8[0]):.2e} mit Faktor 8")
torch.set_printoptions(precision=4, sci_mode=False)

torch.manual_seed(0)
for d in (3, 64, 768):
    q = torch.randn(10_000, d)
    k = torch.randn(10_000, d)
    skalar = (q * k).sum(dim=1)
    print(f"d = {d:3d}: Varianz der Skalarprodukte {skalar.var():7.1f}, nach Division durch √d {(skalar / d**0.5).var():.2f}")

Je größer dₖ, desto weiter streuen die Scores, desto schärfer wird die Softmax und desto schwerer wird das Lernen von Wq und Wk. Die Division durch √dₖ gleicht den Teil der Streuung aus, der durch die Breite dₖ entsteht. Diese Begründung setzt voraus, dass die Einträge zu Beginn des Trainings die Varianz 1 haben. Für gelernte Queries und Keys gilt das nicht automatisch, eine Garantie ist die Skalierung also nicht. GPT-2 teilt je Kopf durch √64 = 8.

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: zwei Klassen für dieselbe Rechnung — einfach ausführen</span>&#10;<span style="color: #008000; font-weight: bold">class</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">SelfAttention_v1</span>(nn<span style="color: #666">.</span>Module):</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Fassung 1 hält drei Matrizen als <code>nn.Parameter</code>, wie in der Zelle oben.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">__init__</span>(<span style="color: #008000">self</span>, d_in, d_out):&#10;        <span style="color: #008000">super</span>()<span style="color: #666">.</span><span style="color: #00F">__init__</span>()&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_query <span style="color: #666">=</span> nn<span style="color: #666">.</span>Parameter(torch<span style="color: #666">.</span>rand(d_in, d_out))&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_key <span style="color: #666">=</span> nn<span style="color: #666">.</span>Parameter(torch<span style="color: #666">.</span>rand(d_in, d_out))&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_value <span style="color: #666">=</span> nn<span style="color: #666">.</span>Parameter(torch<span style="color: #666">.</span>rand(d_in, d_out))</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Der <code>forward</code> rechnet die drei Schritte für alle Tokens auf einmal: Projektionen, Scores als Matrix, skalierte Softmax je Zeile, gewichtete Summe der Values.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">forward</span>(<span style="color: #008000">self</span>, x):&#10;        keys <span style="color: #666">=</span> x <span style="color: #666">@</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_key&#10;        queries <span style="color: #666">=</span> x <span style="color: #666">@</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_query&#10;        values <span style="color: #666">=</span> x <span style="color: #666">@</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_value&#10;        attn_scores <span style="color: #666">=</span> queries <span style="color: #666">@</span> keys<span style="color: #666">.</span>T&#10;        attn_weights <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(attn_scores <span style="color: #666">/</span> keys<span style="color: #666">.</span>shape[<span style="color: #666">-1</span>]<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;        context_vec <span style="color: #666">=</span> attn_weights <span style="color: #666">@</span> values&#10;        <span style="color: #008000; font-weight: bold">return</span> context_vec&#10;&#10;&#10;<span style="color: #008000; font-weight: bold">class</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">SelfAttention_v2</span>(nn<span style="color: #666">.</span>Module):</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Fassung 2 verwendet <code>nn.Linear</code> statt roher Matrizen. Die Rechnung bleibt gleich, aber <code>nn.Linear</code> wählt bessere Startwerte und kann auf Wunsch einen Bias hinzufügen (<code>qkv_bias</code>). GPT-2 nutzt den Bias, das Buch lässt ihn standardmäßig weg.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">__init__</span>(<span style="color: #008000">self</span>, d_in, d_out, qkv_bias<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>):&#10;        <span style="color: #008000">super</span>()<span style="color: #666">.</span><span style="color: #00F">__init__</span>()&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_query <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_in, d_out, bias<span style="color: #666">=</span>qkv_bias)&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_key <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_in, d_out, bias<span style="color: #666">=</span>qkv_bias)&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_value <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_in, d_out, bias<span style="color: #666">=</span>qkv_bias)&#10;&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">forward</span>(<span style="color: #008000">self</span>, x):&#10;        keys <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_key(x)&#10;        queries <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_query(x)&#10;        values <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_value(x)&#10;        attn_scores <span style="color: #666">=</span> queries <span style="color: #666">@</span> keys<span style="color: #666">.</span>T&#10;        attn_weights <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(attn_scores <span style="color: #666">/</span> keys<span style="color: #666">.</span>shape[<span style="color: #666">-1</span>]<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;        context_vec <span style="color: #666">=</span> attn_weights <span style="color: #666">@</span> values&#10;        <span style="color: #008000; font-weight: bold">return</span> context_vec</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>④</b>&nbsp; Dieselben Seeds wie im Buch, damit die Ausgaben mit <code>inputs_buch</code> denen im Buch entsprechen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">123</span>)&#10;sa_v1 <span style="color: #666">=</span> SelfAttention_v1(d_in, d_out)&#10;torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">789</span>)&#10;sa_v2 <span style="color: #666">=</span> SelfAttention_v2(d_in, d_out)&#10;&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"v1:"</span>, sa_v1(inputs)<span style="color: #666">.</span>shape)&#10;<span style="color: #008000">print</span>(sa_v1(inputs))&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"v2:"</span>, sa_v2(inputs)<span style="color: #666">.</span>shape)&#10;<span style="color: #008000">print</span>(sa_v2(inputs))</code></pre></details>

In [ ]:
# Vorgabe: zwei Klassen für dieselbe Rechnung — einfach ausführen
class SelfAttention_v1(nn.Module):
    def __init__(self, d_in, d_out):
        super().__init__()
        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        self.W_value = nn.Parameter(torch.rand(d_in, d_out))

    def forward(self, x):
        keys = x @ self.W_key
        queries = x @ self.W_query
        values = x @ self.W_value
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec


class SelfAttention_v2(nn.Module):
    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

    def forward(self, x):
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)
        attn_scores = queries @ keys.T
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        context_vec = attn_weights @ values
        return context_vec


torch.manual_seed(123)
sa_v1 = SelfAttention_v1(d_in, d_out)
torch.manual_seed(789)
sa_v2 = SelfAttention_v2(d_in, d_out)

print("v1:", sa_v1(inputs).shape)
print(sa_v1(inputs))
print("v2:", sa_v2(inputs).shape)
print(sa_v2(inputs))

Beide Klassen erwarten eine einzelne Folge der Form `[T, d_in]`. Das erkennt man an `keys.T`. Eine Batch-Dimension kommt erst in Kapitel 3.3 dazu. Die Ausgaben unterscheiden sich, weil die Startwerte verschieden sind, nicht weil die Rechnung eine andere wäre. Das zeigt die erste Übung.

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E75B6; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Was gelernt wird und was nicht</div>
<div style="background:#EAF2FA; color:#1a1a1a; padding:12px 16px; margin:0;">
<p><strong>Parameter</strong> sind <code>W_query</code>, <code>W_key</code>, <code>W_value</code> (mit ihren Bias-Vektoren, falls eingeschaltet) und später die Ausgabeprojektion. Sie sind für jeden Satz gleich.</p>
<p><strong>Keine Parameter</strong> sind Scores, Gewichte und Kontextvektoren. Sie werden für jeden Satz neu berechnet und hängen trotzdem vom Gelernten ab: die Gewichte über Q und K, die Kontextvektoren zusätzlich über V. Bei dieser Implementierung behält PyTorch die Gewichtsmatrix für den Rückwärtsschritt, solange Gradienten aufgezeichnet werden. Bei der Textgenerierung unter <code>torch.no_grad()</code> entfällt das, und effiziente Attention-Kerne speichern sie auch im Training nicht vollständig.</p>

</div>
</div>

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">3.2.1&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">8 min</span>🔬 Aufgabe 3.2 #1 · Experiment · Hausaufgabe, nach Wahl</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie sehen, wie die beiden Klassen zusammenhängen und an welcher Stelle der Rechnung sich ein Bias überhaupt auswirkt.</em></p>
<p><code>SelfAttention_v1</code> und <code>SelfAttention_v2</code> rechnen dasselbe, nur sind die Matrizen anders abgelegt: <code>nn.Linear</code> speichert die Matrix transponiert als <code>weight</code> der Form <code>[d_out, d_in]</code>. Die Zelle überträgt die drei Matrizen von <code>sa_v2</code> nach <code>sa_v1</code> (Raschka, Übung 3.1) und legt eine dritte Instanz mit Bias an, in der zunächst alle Bias-Vektoren 0 sind.</p>
<p>Mit <code>bias_auf</code> wählen Sie, welcher Bias auf 0,5 gesetzt wird. Beginnen Sie mit <code>"value"</code>, dann <code>"key"</code>, dann <code>"query"</code>. Sagen Sie für jeden Fall vorher: Ändern sich die Gewichte, die Kontextvektoren, beides oder nichts?</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Die Zelle gibt Gewichte und Kontextvektoren mit und ohne Bias nebeneinander aus. Für die Keys lohnt sich ein Blick auf die Formel: qᵀ(k + b) = qᵀk + qᵀb.</p></div></details>

</div>
</div>

In [ ]:
# Teil 1: Matrizen von v2 nach v1 übertragen (weight hat die Form [d_out, d_in], deshalb .T)
sa_v1.W_query = nn.Parameter(sa_v2.W_query.weight.T.clone())
sa_v1.W_key = nn.Parameter(sa_v2.W_key.weight.T.clone())
sa_v1.W_value = nn.Parameter(sa_v2.W_value.weight.T.clone())
print("v1 und v2 gleich:", torch.allclose(sa_v1(inputs), sa_v2(inputs), atol=1e-5))

# Teil 2: eine Instanz mit Bias, alle Bias-Vektoren zunächst 0
sa_v2_bias = SelfAttention_v2(d_in, d_out, qkv_bias=True)
sa_v2_bias.load_state_dict(sa_v2.state_dict(), strict=False)      # Matrizen übernehmen
with torch.no_grad():
    for schicht in (sa_v2_bias.W_query, sa_v2_bias.W_key, sa_v2_bias.W_value):
        schicht.bias.zero_()

vorhersage = "..."             # "gewichte", "kontext", "beides" oder "nichts"
bias_auf = "value"             # hier ändern: "value", "key", "query"


def gewichte_von(modul, x):
    """Attention-Gewichte eines SelfAttention-Moduls, zum Vergleichen."""
    with torch.no_grad():
        scores = modul.W_query(x) @ modul.W_key(x).T
        return torch.softmax(scores / modul.W_key(x).shape[-1]**0.5, dim=-1)


with torch.no_grad():
    getattr(sa_v2_bias, "W_" + bias_auf).bias.fill_(0.5)
    gewichte_gleich = torch.allclose(gewichte_von(sa_v2_bias, inputs), gewichte_von(sa_v2, inputs), atol=1e-6)
    kontext_gleich = torch.allclose(sa_v2_bias(inputs), sa_v2(inputs), atol=1e-6)
    getattr(sa_v2_bias, "W_" + bias_auf).bias.zero_()               # zurücksetzen für den nächsten Versuch
print(f"Bias auf {bias_auf}: Gewichte unverändert: {gewichte_gleich}, Kontextvektoren unverändert: {kontext_gleich}")

In [ ]:
# Prüfzelle — einfach ausführen
assert torch.allclose(sa_v1(inputs), sa_v2(inputs), atol=1e-5), "v1 und v2 sollten nach der Übertragung dieselbe Ausgabe liefern"
assert vorhersage in ("gewichte", "kontext", "beides", "nichts"), "Tragen Sie gewichte, kontext, beides oder nichts ein"
_erwartet = {"value": "kontext", "key": "nichts", "query": "beides"}[bias_auf]
print(f"Bias auf {bias_auf}: beobachtet {_erwartet!r} — Ihre Vorhersage: {vorhersage!r}")
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">7 min</span>🔬 Aufgabe 3.2 #2 · Experiment · Hausaufgabe, Pflicht</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie sehen, wovon die Schärfe der Softmax abhängt und was die Division durch √d dabei leistet und was nicht.</em></p>
<p>Die Zelle erzeugt mit <code>torch.randn</code> Queries und Keys für 50 Tokens, jeder Eintrag hat Mittelwert 0 und Varianz 1. Dann berechnet sie die Gewichte der ersten Zeile, einmal mit und einmal ohne Division durch √d. Gezählt wird, wie viele der 50 Tokens noch ein Gewicht über 0,01 bekommen.</p>
<p>Schreiben Sie vorab in <code>vorhersage_ohne_768</code>, wie viele Tokens es bei d = 768 ohne Division sind. Führen Sie die Zelle aus, dann setzen Sie <code>skala</code> auf 0,1 (Einträge mit kleinerer Varianz) und sehen nach, was mit und ohne Division passiert. Erklären Sie danach in zwei Sätzen als Kommentar, wie Varianz der Einträge, Streuung der Scores und Softmax zusammenhängen.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Die Streuung der Scores wächst mit der Breite d und mit der Varianz der Einträge. Die Softmax reagiert auf Abstände zwischen Scores.</p></div></details>

</div>
</div>

In [ ]:
vorhersage_ohne_768 = ...    # Zahl zwischen 1 und 50
skala = 1.0                  # Standardabweichung der Einträge, danach 0.1 probieren

torch.manual_seed(1)
ergebnis = {}
for d in (64, 768):
    q = skala * torch.randn(50, d)
    k = skala * torch.randn(50, d)
    scores = q @ k.T
    gewichte_mit = torch.softmax(scores / d**0.5, dim=-1)
    gewichte_ohne = torch.softmax(scores, dim=-1)
    ergebnis[d] = (int((gewichte_mit[0] > 0.01).sum()), int((gewichte_ohne[0] > 0.01).sum()))
    print(f"d = {d:3d}, skala {skala}: Varianz der Scores {scores.var():8.1f}, mit Division {ergebnis[d][0]:2d} Tokens über 0.01, ohne {ergebnis[d][1]:2d}")

# Erklärung (zwei Sätze):
# ...

In [ ]:
# Prüfzelle — einfach ausführen
assert isinstance(vorhersage_ohne_768, int) and 1 <= vorhersage_ohne_768 <= 50, "Tragen Sie eine Zahl zwischen 1 und 50 ein"
if skala == 1.0:
    assert ergebnis[768][1] <= 3, "Ohne Division sollte bei d = 768 fast alles Gewicht auf einem Token liegen"
    assert ergebnis[768][0] >= 10, "Mit Division verteilt sich das Gewicht auf viele Tokens"
print("Ihre Vorhersage:", vorhersage_ohne_768, "— gemessen bei skala 1:", ergebnis[768][1] if skala == 1.0 else "(skala geändert, siehe Ausgabe oben)")
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 3.2 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Wo stecken in <code>SelfAttention_v2</code> die Parameter, und wie viele sind es bei <code>d_in = 3</code>, <code>d_out = 2</code> ohne Bias?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Ein Bias auf den <strong>Keys</strong> ändert die Gewichte nicht. Warum? (Rechnen Sie qᵀ(k + b) aus.)</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Warum skaliert GPT-2 mit √64 und nicht mit √768?</p>

</div>

## <a id="kap-causal-attention"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;">3.3&nbsp;&nbsp;Causal Attention</span>

In Kapitel 3.1 gab „nach“ sein höchstes Gewicht an „Worms“, ein Token, das erst später im Satz steht. Beim Training eines Sprachmodells ist das Ziel für Position i das Token i + 1 (Lerneinheit 2). Dürfte Position i auf spätere Positionen schauen, könnte sie die richtige Antwort direkt aus der Eingabe ablesen. Die **Causal Mask** verbietet das: Jede Position sieht nur sich selbst und ihre Vorgänger (Raschka, Abschnitt 3.5). Zunächst bauen wir die Maske Schritt für Schritt mit einzelnen Tensoren, noch ohne Klasse. Die Klasse dazu schreiben Sie in der ersten Übung selbst.

Die Zahlen hier stammen aus den projizierten und skalierten Scores von `sa_v2` (Kapitel 3.2). Die Folien zeigen dieselbe Maske am Beispiel ohne Parameter aus Kapitel 3.1, deshalb sehen die Gewichte dort anders aus. Der Mechanismus ist derselbe.

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Queries, Keys und Values kommen aus <code>sa_v2</code>, damit die Zahlen zu Kapitel 3.2 passen. <code>torch.no_grad</code> schaltet die Gradientenaufzeichnung ab, weil wir hier nicht trainieren.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">with</span> torch<span style="color: #666">.</span>no_grad():&#10;    queries <span style="color: #666">=</span> sa_v2<span style="color: #666">.</span>W_query(inputs)&#10;    keys <span style="color: #666">=</span> sa_v2<span style="color: #666">.</span>W_key(inputs)&#10;    values <span style="color: #666">=</span> sa_v2<span style="color: #666">.</span>W_value(inputs)&#10;d_k <span style="color: #666">=</span> keys<span style="color: #666">.</span>shape[<span style="color: #666">-1</span>]&#10;&#10;attn_scores <span style="color: #666">=</span> queries <span style="color: #666">@</span> keys<span style="color: #666">.</span>T&#10;attn_weights <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(attn_scores <span style="color: #666">/</span> d_k<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Gewichte ohne Maske"</span>)&#10;zeige(attn_weights, woerter, woerter)</code></pre></details>

In [ ]:
# Vorgabe: Gewichte ohne Maske — einfach ausführen
with torch.no_grad():
    queries = sa_v2.W_query(inputs)
    keys = sa_v2.W_key(inputs)
    values = sa_v2.W_value(inputs)
d_k = keys.shape[-1]

attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / d_k**0.5, dim=-1)
print("Gewichte ohne Maske")
zeige(attn_weights, woerter, woerter)

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: Weg 1, Nullen und Renormieren — einfach ausführen</span>&#10;T <span style="color: #666">=</span> attn_scores<span style="color: #666">.</span>shape[<span style="color: #666">0</span>]</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; <code>tril</code> behält die untere Dreiecksmatrix einschließlich Diagonale: 1 heißt erlaubt, 0 heißt Zukunft.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>mask_simple <span style="color: #666">=</span> torch<span style="color: #666">.</span>tril(torch<span style="color: #666">.</span>ones(T, T))&#10;<span style="color: #008000">print</span>(mask_simple)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Wir setzen die Gewichte oberhalb der Diagonalen auf null. Danach summieren sich die Zeilen nicht mehr zu 1.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>masked_simple <span style="color: #666">=</span> attn_weights <span style="color: #666">*</span> mask_simple</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Danach teilen wir jede Zeile durch ihre Summe, sodass sie sich wieder zu 1 summiert.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>weg1 <span style="color: #666">=</span> masked_simple <span style="color: #666">/</span> masked_simple<span style="color: #666">.</span>sum(dim<span style="color: #666">=-1</span>, keepdim<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>)&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Weg 1: maskiert und renormiert"</span>)&#10;zeige(weg1, woerter, woerter)</code></pre></details>

In [ ]:
# Vorgabe: Weg 1, Nullen und Renormieren — einfach ausführen
T = attn_scores.shape[0]
mask_simple = torch.tril(torch.ones(T, T))
print(mask_simple)

masked_simple = attn_weights * mask_simple
weg1 = masked_simple / masked_simple.sum(dim=-1, keepdim=True)
print()
print("Weg 1: maskiert und renormiert")
zeige(weg1, woerter, woerter)

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Diesmal nehmen wir die obere Dreiecksmatrix ohne Diagonale (<code>diagonal=1</code>): 1 heißt verboten.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>mask <span style="color: #666">=</span> torch<span style="color: #666">.</span>triu(torch<span style="color: #666">.</span>ones(T, T), diagonal<span style="color: #666">=1</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Verbotene Scores werden auf −∞ gesetzt. exp(−∞) = 0, also bekommen sie in der Softmax das Gewicht 0, und die Zeile summiert sich von selbst zu 1.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>masked <span style="color: #666">=</span> attn_scores<span style="color: #666">.</span>masked_fill(mask<span style="color: #666">.</span>bool(), <span style="color: #666">-</span>torch<span style="color: #666">.</span>inf)&#10;<span style="color: #008000">print</span>(masked)&#10;weg2 <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(masked <span style="color: #666">/</span> d_k<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Weg 2: −∞ vor der Softmax"</span>)&#10;zeige(weg2, woerter, woerter)&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Beide Wege gleich:"</span>, torch<span style="color: #666">.</span>allclose(weg1, weg2, atol<span style="color: #666">=1e-6</span>))</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Die Kontextvektoren mit Maske sind die Referenz für die Bau-Aufgabe unten.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>kontext_causal <span style="color: #666">=</span> weg2 <span style="color: #666">@</span> values</code></pre></details>

In [ ]:
# Vorgabe: Weg 2, minus unendlich vor der Softmax — einfach ausführen
mask = torch.triu(torch.ones(T, T), diagonal=1)
masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
print(masked)
weg2 = torch.softmax(masked / d_k**0.5, dim=-1)
print()
print("Weg 2: −∞ vor der Softmax")
zeige(weg2, woerter, woerter)
print()
print("Beide Wege gleich:", torch.allclose(weg1, weg2, atol=1e-6))

kontext_causal = weg2 @ values

Algebraisch liefern beide Wege dieselben Zahlen, solange in jeder Zeile mindestens eine Position erlaubt ist. Das ist hier immer der Fall, weil die Diagonale erlaubt bleibt. Numerisch gilt das nicht immer: Bei sehr großen Scores kann Weg 1 null durch null teilen, Weg 2 nicht. Aufgabe 3.3 #3 zeigt den Fall. Deshalb ist Weg 2 der übliche. Vaswani et al. 2017 beschreiben ihn in Abschnitt 3.2.3.

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Die nächste Zelle baut absichtlich einen typischen Fehler ein: Sie setzt die verbotenen Scores nicht auf −∞, sondern auf 0. Legen Sie sich fest: <strong>Was passiert mit den Gewichten? Gibt es eine Fehlermeldung?</strong></p>

</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; exp(0) = 1, nicht 0. Die verbotenen Positionen bekommen also Gewicht, als hätten sie den Score 0.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>falsch <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(attn_scores<span style="color: #666">.</span>masked_fill(mask<span style="color: #666">.</span>bool(), <span style="color: #666">0.0</span>) <span style="color: #666">/</span> d_k<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;zeige(falsch, woerter, woerter)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Die Summe der Gewichte oberhalb der Diagonalen misst das Leck, also wie viel Information aus der Zukunft in die Vorhersage fließt.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000">print</span>()&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">f"Gewicht auf der Zukunft insgesamt: </span><span style="color: #A45A77; font-weight: bold">{</span>(falsch<span style="color: #BBB"> </span><span style="color: #666">*</span><span style="color: #BBB"> </span>mask)<span style="color: #666">.</span>sum()<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.2f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> (richtig wäre 0)"</span>)</code></pre></details>

In [ ]:
# Vorgabe: der stille Fehler — einfach ausführen
falsch = torch.softmax(attn_scores.masked_fill(mask.bool(), 0.0) / d_k**0.5, dim=-1)
zeige(falsch, woerter, woerter)
print()
print(f"Gewicht auf der Zukunft insgesamt: {(falsch * mask).sum():.2f} (richtig wäre 0)")

PyTorch meldet dabei weder einen Fehler noch eine Warnung. Das Modell schaut aber beim Training in die Zukunft und erzeugt später deutlich schlechteren Text, als sein Trainings-Loss erwarten lässt. Beim Lesen des Codes übersieht man solche Lecks leicht. Ein Test, der das Verhalten prüft, findet sie dagegen zuverlässig. Die Prüfzelle der Bau-Aufgabe enthält einen solchen Test.

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Beim Dropout werden während des Trainings zufällig ausgewählte Gewichte auf null gesetzt und die übrigen mit 1/(1 − p) hochskaliert. Die nächste Zelle wendet Dropout mit p = 0,5 auf die maskierte Gewichtsmatrix an. Legen Sie sich fest: <strong>Welche Werte stehen danach in der Matrix, und wie viele Nullen sind es ungefähr?</strong></p>

</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: Dropout auf die Gewichte — einfach ausführen</span>&#10;torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">123</span>)&#10;dropout <span style="color: #666">=</span> nn<span style="color: #666">.</span>Dropout(<span style="color: #666">0.5</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Zuerst wenden wir Dropout auf eine Matrix aus Einsen an. Etwa die Hälfte wird 0, der Rest wird 2, damit die Summe im Mittel gleich bleibt.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000">print</span>(dropout(torch<span style="color: #666">.</span>ones(<span style="color: #666">6</span>, <span style="color: #666">6</span>)))</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Dann wenden wir Dropout auf die maskierten Gewichte an. Die 15 Nullen der Maske bleiben, und von den 21 erlaubten Einträgen fällt im Mittel die Hälfte weg. Zusammen sind rund 71 % der Matrix null.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">123</span>)&#10;nach_dropout <span style="color: #666">=</span> dropout(weg2)&#10;zeige(nach_dropout, woerter, woerter)&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">f"Anteil Nullen: </span><span style="color: #A45A77; font-weight: bold">{</span>(nach_dropout<span style="color: #BBB"> </span><span style="color: #666">==</span><span style="color: #BBB"> </span><span style="color: #666">0</span>)<span style="color: #666">.</span>float()<span style="color: #666">.</span>mean()<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">.0%</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">"</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Zeilensummen:"</span>, nach_dropout<span style="color: #666">.</span>sum(dim<span style="color: #666">=-1</span>))</code></pre></details>

In [ ]:
# Vorgabe: Dropout auf die Gewichte — einfach ausführen
torch.manual_seed(123)
dropout = nn.Dropout(0.5)

print(dropout(torch.ones(6, 6)))

torch.manual_seed(123)
nach_dropout = dropout(weg2)
zeige(nach_dropout, woerter, woerter)
print()
print(f"Anteil Nullen: {(nach_dropout == 0).float().mean():.0%}")
print("Zeilensummen:", nach_dropout.sum(dim=-1))

Nach dem Dropout summieren sich die Zeilen nicht mehr genau zu 1, sondern nur noch im Erwartungswert. Der Zweck ist Regularisierung: Das Modell kann sich nicht darauf verlassen, dass ein bestimmtes Gewicht immer da ist, und lernt robustere Mischungen. Raschka verwendet p = 0,5, damit der Effekt deutlich sichtbar wird. GPT-2 nutzt 0,1 (`attn_pdrop` in der Konfiguration). Bei der Textgenerierung ist Dropout abgeschaltet, dafür sorgt `model.eval()`. In der Klasse unten braucht Dropout nur eine Zeile, mehr müssen Sie dazu in dieser Lerneinheit nicht wissen.

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Wir stapeln zwei Kopien des Satzes zu einem Batch der Form [B, T, d_in]. Ab jetzt arbeiten alle Rechnungen mit dieser Form.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>batch <span style="color: #666">=</span> torch<span style="color: #666">.</span>stack((inputs, inputs), dim<span style="color: #666">=0</span>)&#10;<span style="color: #008000">print</span>(batch<span style="color: #666">.</span>shape)</code></pre></details>

In [ ]:
# Vorgabe: ein Batch — einfach ausführen
batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

Mit einer Batch-Dimension funktioniert `keys.T` nicht mehr, denn `.T` würde die Reihenfolge aller drei Achsen umkehren. Stattdessen brauchen wir `keys.transpose(1, 2)`. Das vertauscht nur die Achsen für Tokens und Dimensionen, der Batch bleibt vorn. Das ist einer der Unterschiede zwischen der Rechnung mit einzelnen Tensoren oben und der Klasse, die Sie jetzt bauen.

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">3.3.1&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#B4532A; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">10 min</span>🔧 Aufgabe 3.3 #1 · Bauen · in der Vorlesung</div>
<div style="background:#FBEDE6; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie fassen die Schritte des ersten Teils zu einer Klasse zusammen und testen, ob die Klasse wirklich kausal ist. Diese Klasse ist der Kern von <code>MultiHeadAttention</code> und damit von jedem GPT-Block.</em></p>
<p>Setzen Sie die Schritte dieses Kapitels zur Klasse <code>CausalAttention</code> zusammen. Konstruktor und Prüfzelle stehen, im <code>forward</code> fehlen vier Zeilen:</p>
<ol>
<li>Scores für den Batch (<code>keys</code> hat die Form <code>[B, T, d_out]</code>, <code>.T</code> geht hier nicht).</li>
<li>Zukunft ausblenden: die Maske auf die ersten <code>num_tokens</code> Positionen zuschneiden und −∞ einsetzen.</li>
<li>Softmax mit Skalierung, je Zeile.</li>
<li>Dropout auf die Gewichte, dann die gewichtete Summe der Values.</li>
</ol>
<p>Maske und Skalierung müssen beide vor der Softmax stehen. Ihre Reihenfolge untereinander ist egal (warum?). Die Prüfzelle vergleicht Ihr Ergebnis mit <code>kontext_causal</code> von oben und testet außerdem das Verhalten: Wenn sich ein späteres Token ändert, dürfen sich die Ausgaben an früheren Positionen nicht ändern. Das ist Kausalität, und sie muss für jede Wahl der Gewichte gelten. Ein Gegentest prüft mit den konkreten Gewichten, dass ein geändertes früheres Token die Ausgabe an Position 3 sehr wohl verändert. Andernfalls wäre die Maske zu streng.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#B4532A; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #B4532A55"><p><strong>Hinweis:</strong> <code>queries @ keys.transpose(1, 2)</code>, dann <code>attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)</code>. Die Maske ist als Puffer angelegt (<code>register_buffer</code>). Sie ist kein Parameter, wird aber zusammen mit dem Modell auf die GPU verschoben.</p></div></details>

</div>
</div>

In [ ]:
class CausalAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()
        self.d_out = d_out
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)
        queries = self.W_query(x)
        values = self.W_value(x)

        # TODO Lücke 1: Scores für den Batch, Form [B, T, T]
        attn_scores = ...
        # TODO Lücke 2: Zukunft ausblenden (Maske zuschneiden, −∞ einsetzen)
        ...
        # TODO Lücke 3: Softmax mit Skalierung, je Zeile
        attn_weights = ...
        # TODO Lücke 4: Dropout, dann gewichtete Summe der Values
        attn_weights = ...
        context_vec = ...
        return context_vec

In [ ]:
# Prüfzelle — einfach ausführen
ca = CausalAttention(d_in, d_out, context_length=6, dropout=0.0)
ca.load_state_dict(sa_v2.state_dict(), strict=False)   # dieselben Matrizen wie oben, Dropout aus
ca.eval()
with torch.no_grad():
    out = ca(batch)
    assert out.shape == (2, 6, d_out), f"Ausgabeform sollte (2, 6, {d_out}) sein, ist {tuple(out.shape)}"
    assert torch.allclose(out[0], kontext_causal, atol=1e-5), "Die Ausgabe sollte den Kontextvektoren mit Maske aus dem ersten Teil entsprechen"
    # Verhalten 1: ein späteres Token darf frühere Ausgaben nicht ändern
    spaeter = batch.clone(); spaeter[:, 5] = torch.tensor([1.0, -1.0, 0.5])
    assert torch.allclose(ca(spaeter)[:, :5], out[:, :5], atol=1e-6), "Leck: eine Änderung an Position 5 verändert frühere Ausgaben. Prüfen Sie die Maske."
    # Gegentest mit diesen Gewichten: ein früheres Token erreicht Position 3 (sonst wäre die Maske zu streng)
    frueher = batch.clone(); frueher[:, 0] = torch.tensor([1.0, -1.0, 0.5])
    assert not torch.allclose(ca(frueher)[:, 3], out[:, 3], atol=1e-4), "Position 3 sollte Position 0 sehen. Prüfen Sie, ob die Maske nur oberhalb der Diagonalen liegt (diagonal=1)"
    # Verhalten 3: die erste Position sieht nur sich selbst
    assert torch.allclose(out[:, 0], ca.W_value(batch)[:, 0], atol=1e-5), "Position 0 darf nur ihren eigenen Value sehen (Gewicht 1 auf sich selbst)"
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#B4532A; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">10 min</span>🔧 Aufgabe 3.3 #2 · Bauen · Hausaufgabe, nach Wahl</div>
<div style="background:#FBEDE6; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie sehen, wie sich mit der Maske die Reichweite einer Position festlegen lässt. Gemma 3 und gpt-oss verwenden solche Fensterschichten. Dort sorgen spezielle Kerne dafür, dass tatsächlich Rechenzeit und Speicher gespart werden.</em></p>
<p>Erweitern Sie Ihre Klasse um einen Parameter <code>fenster</code>: Position i sieht nur die Positionen j mit i − fenster + 1 ≤ j ≤ i, also sich selbst und die <code>fenster − 1</code> Vorgänger (lokale Attention, Vaswani et al. 2017 nennen sie „restricted“ in Tabelle 1). Ist <code>fenster</code> mindestens so groß wie T, muss sich die Klasse wie bisher verhalten. Die Prüfzelle testet zweierlei: Eine Änderung außerhalb des Fensters von Position i lässt die Ausgabe an i unverändert, und eine Änderung innerhalb ändert sie (mit diesen Gewichten).</p>
<p>Hinweis zum Aufwand: Diese Fassung rechnet weiterhin alle T × T Scores und maskiert nur mehr davon. Rechenzeit und Speicher spart lokale Attention erst mit einer Implementierung, die die Scores außerhalb des Fensters gar nicht bildet. Die Aufgabe zeigt also nur, wie sich das Fenster auf die Reichweite auswirkt.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#B4532A; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #B4532A55"><p><strong>Hinweis:</strong> Zur Causal Mask kommt eine zweite Dreiecksmatrix: <code>torch.tril(torch.ones(T, T), diagonal=-fenster)</code> markiert alles, was zu weit zurückliegt. Beide Masken mit <code>+</code> oder <code>|</code> verbinden.</p></div></details>

</div>
</div>

In [ ]:
class CausalAttentionFenster(CausalAttention):
    def __init__(self, d_in, d_out, context_length, dropout, fenster, qkv_bias=False):
        super().__init__(d_in, d_out, context_length, dropout, qkv_bias)
        # TODO: Maske ersetzen, sodass auch Positionen vor i − fenster + 1 verboten sind
        ...

In [ ]:
# Prüfzelle — einfach ausführen
caf = CausalAttentionFenster(d_in, d_out, context_length=6, dropout=0.0, fenster=2)
caf.load_state_dict(sa_v2.state_dict(), strict=False)
caf.eval()
ca_voll = CausalAttentionFenster(d_in, d_out, context_length=6, dropout=0.0, fenster=6)
ca_voll.load_state_dict(sa_v2.state_dict(), strict=False)
ca_voll.eval()
with torch.no_grad():
    assert torch.allclose(ca_voll(batch)[0], kontext_causal, atol=1e-5), "Mit fenster >= T muss alles wie bei der vollen Causal Attention sein"
    out = caf(batch)
    weit = batch.clone(); weit[:, 0] = torch.tensor([1.0, -1.0, 0.5])        # Position 0 liegt für Position 3 außerhalb des Fensters 2
    assert torch.allclose(caf(weit)[:, 3], out[:, 3], atol=1e-6), "Position 3 darf bei fenster=2 nicht auf Position 0 reagieren"
    nah = batch.clone(); nah[:, 2] = torch.tensor([1.0, -1.0, 0.5])          # Position 2 liegt für Position 3 im Fenster
    assert not torch.allclose(caf(nah)[:, 3], out[:, 3], atol=1e-4), "Position 3 muss bei fenster=2 auf Position 2 reagieren"
    assert torch.allclose(out[:, 0], caf.W_value(batch)[:, 0], atol=1e-5), "Position 0 sieht weiterhin nur sich selbst"
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">8 min</span>🔬 Aufgabe 3.3 #3 · Experiment · Hausaufgabe, nach Wahl</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie sehen, worin sich die beiden Maskenwege unterscheiden, obwohl sie algebraisch dasselbe berechnen, und was ein falscher Maskenwert anrichtet.</em></p>
<p>Beide Maskenwege stehen als Funktionen fertig da. Die Variable <code>erlaubt</code> ist eine Bool-Matrix, <code>True</code> steht für eine erlaubte Position. Es gibt drei Versuche. Notieren Sie jeweils zuerst Ihre Vorhersage und führen Sie dann die Zelle aus:</p>
<ol>
<li>Normale Scores (<code>alle_scores</code> aus Kapitel 3.1, Causal Mask): Liefern beide Wege dasselbe? (<code>vorhersage_1</code>: "gleich" oder "verschieden")</li>
<li>Extremfall in float32: Scores <code>[0, 1000]</code>, nur der erste Eintrag erlaubt. Was liefert Weg 1, was Weg 2? (<code>vorhersage_2</code>: Ihre Erwartung als Text)</li>
<li>Setzen Sie <code>maskenwert</code> von <code>-torch.inf</code> auf <code>0.0</code>. Wie viel Gewicht landet dann insgesamt auf verbotenen Positionen? (<code>vorhersage_3</code>: eine Zahl)</li>
</ol>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Weg 1 rechnet die Softmax über alle Scores, auch die verbotenen, und repariert danach. Überlegen Sie, was exp(1000) in float32 ergibt.</p></div></details>

</div>
</div>

In [ ]:
def maskiere_weg1(scores, erlaubt):
    gewichte = torch.softmax(scores, dim=-1) * erlaubt
    return gewichte / gewichte.sum(dim=-1, keepdim=True)


def maskiere_weg2(scores, erlaubt, maskenwert=-torch.inf):
    return torch.softmax(scores.masked_fill(~erlaubt, maskenwert), dim=-1)


vorhersage_1 = "..."         # "gleich" oder "verschieden"
vorhersage_2 = "..."         # was liefern Weg 1 und Weg 2 bei [0, 1000]?
vorhersage_3 = ...           # Zahl: Gewicht auf verbotenen Positionen, wenn 0 statt -inf eingesetzt wird
maskenwert = -torch.inf      # für Versuch 3 auf 0.0 setzen

erlaubt = torch.tril(torch.ones(6, 6)).bool()
print("1. gleich:", torch.allclose(maskiere_weg1(alle_scores, erlaubt), maskiere_weg2(alle_scores, erlaubt), atol=1e-6))

extrem = torch.tensor([[0.0, 1000.0]])
nur_erster = torch.tensor([[True, False]])
print("2. Weg 1:", maskiere_weg1(extrem, nur_erster), " Weg 2:", maskiere_weg2(extrem, nur_erster))

leck = maskiere_weg2(alle_scores, erlaubt, maskenwert)
leck_summe = (leck * ~erlaubt).sum()
print(f"3. Gewicht auf verbotenen Positionen bei maskenwert {maskenwert}: {leck_summe:.2f}")

In [ ]:
# Prüfzelle — einfach ausführen
assert vorhersage_1 in ("gleich", "verschieden") and isinstance(vorhersage_2, str) and vorhersage_2 != "..." and not isinstance(vorhersage_3, type(...)), "Tragen Sie alle drei Vorhersagen ein"
assert torch.isnan(maskiere_weg1(extrem, nur_erster)).any(), "Weg 1 liefert im Extremfall NaN (0 geteilt durch 0)"
assert torch.allclose(maskiere_weg2(extrem, nur_erster), torch.tensor([[1.0, 0.0]])), "Weg 2 liefert im Extremfall [1, 0]"
assert maskenwert == 0.0, "Setzen Sie maskenwert für Versuch 3 auf 0.0"
assert float(leck_summe) > 1.0, "Mit 0 statt -inf sollte deutlich Gewicht auf der Zukunft liegen (rund 2 über alle sechs Zeilen)"
print(f"Beobachtet: 1. gleich, 2. NaN gegen [1, 0], 3. Leck {float(leck_summe):.2f} — Ihre Vorhersagen: {vorhersage_1!r}, {vorhersage_2!r}, {vorhersage_3}")
print("Alle Prüfungen bestanden ✓")

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 3.3 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Warum ist die Maske ein Puffer (<code>register_buffer</code>) und kein Parameter?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Warum wird die Maske im <code>forward</code> auf <code>num_tokens</code> zugeschnitten, statt sie gleich in der passenden Größe anzulegen?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Mit Maske löst ein einziger Durchlauf über einen Satz der Länge T alle T Vorhersageaufgaben gleichzeitig. Warum lässt sich das parallel rechnen, und warum ist es zulässig?</p>

</div>

## <a id="kap-multi-head-attention"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;">3.4&nbsp;&nbsp;Multi-Head Attention</span>

Ein einzelner Kopf berechnet je Query eine Gewichtsverteilung und mischt Values aus einem einzigen Wertraum. Mehrere Köpfe berechnen getrennte Verteilungen in getrennten Unterräumen. Dadurch kann eine Position für verschiedene Zwecke auf verschiedene Stellen im Satz schauen (Vaswani et al. 2017, Abschnitt 3.2.2). Raschka baut das zweimal: zuerst als Liste von Köpfen (`MultiHeadAttentionWrapper`), dann als eine Klasse mit einer großen Matrix, die per `view` und `transpose` in Köpfe aufgeteilt wird. Die zweite Fassung ist schneller und wird in Lerneinheit 4 unverändert in das GPT-Modell eingebaut. Sie ist hier vollständig vorgegeben.

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: Multi-Head Attention — einfach ausführen</span>&#10;<span style="color: #008000; font-weight: bold">class</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">MultiHeadAttention</span>(nn<span style="color: #666">.</span>Module):&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">__init__</span>(<span style="color: #008000">self</span>, d_in, d_out, context_length, dropout, num_heads, qkv_bias<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>):&#10;        <span style="color: #008000">super</span>()<span style="color: #666">.</span><span style="color: #00F">__init__</span>()&#10;        <span style="color: #008000; font-weight: bold">assert</span> d_out <span style="color: #666">%</span> num_heads <span style="color: #666">==</span> <span style="color: #666">0</span>, <span style="color: #BA2121">"d_out muss durch num_heads teilbar sein"</span>&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>d_out <span style="color: #666">=</span> d_out&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>num_heads <span style="color: #666">=</span> num_heads</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; <code>d_out</code> ist die Gesamtbreite. Jeder Kopf bekommt <code>head_dim = d_out / num_heads</code> Dimensionen, in GPT-2 small 768 / 12 = 64.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        <span style="color: #008000">self</span><span style="color: #666">.</span>head_dim <span style="color: #666">=</span> d_out <span style="color: #666">//</span> num_heads&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_query <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_in, d_out, bias<span style="color: #666">=</span>qkv_bias)&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_key <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_in, d_out, bias<span style="color: #666">=</span>qkv_bias)&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>W_value <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_in, d_out, bias<span style="color: #666">=</span>qkv_bias)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Neu gegenüber <code>CausalAttention</code> ist die Ausgabeprojektion, die die zusammengefügten Köpfe noch einmal mischt. Sie hat immer einen Bias.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        <span style="color: #008000">self</span><span style="color: #666">.</span>out_proj <span style="color: #666">=</span> nn<span style="color: #666">.</span>Linear(d_out, d_out)&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>dropout <span style="color: #666">=</span> nn<span style="color: #666">.</span>Dropout(dropout)&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>register_buffer(<span style="color: #BA2121">"mask"</span>, torch<span style="color: #666">.</span>triu(torch<span style="color: #666">.</span>ones(context_length, context_length), diagonal<span style="color: #666">=1</span>))&#10;&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">forward</span>(<span style="color: #008000">self</span>, x, return_weights<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>):&#10;        b, num_tokens, d_in <span style="color: #666">=</span> x<span style="color: #666">.</span>shape&#10;        keys <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_key(x)          <span style="color: #3D7B7B; font-style: italic"># [B, T, d_out]</span>&#10;        queries <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_query(x)&#10;        values <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>W_value(x)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Aufteilen in Köpfe: <code>view</code> zerlegt die letzte Dimension d_out in (num_heads, head_dim), ohne Daten zu kopieren. Die Form ist danach [B, T, h, d_h].</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        keys <span style="color: #666">=</span> keys<span style="color: #666">.</span>view(b, num_tokens, <span style="color: #008000">self</span><span style="color: #666">.</span>num_heads, <span style="color: #008000">self</span><span style="color: #666">.</span>head_dim)&#10;        values <span style="color: #666">=</span> values<span style="color: #666">.</span>view(b, num_tokens, <span style="color: #008000">self</span><span style="color: #666">.</span>num_heads, <span style="color: #008000">self</span><span style="color: #666">.</span>head_dim)&#10;        queries <span style="color: #666">=</span> queries<span style="color: #666">.</span>view(b, num_tokens, <span style="color: #008000">self</span><span style="color: #666">.</span>num_heads, <span style="color: #008000">self</span><span style="color: #666">.</span>head_dim)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>④</b>&nbsp; <code>transpose(1, 2)</code> bringt die Köpfe vor die Tokens: [B, h, T, d_h]. Jetzt rechnet die Matrixmultiplikation je Kopf, als wären es h unabhängige Attention-Schichten.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        keys <span style="color: #666">=</span> keys<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)&#10;        queries <span style="color: #666">=</span> queries<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)&#10;        values <span style="color: #666">=</span> values<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>⑤</b>&nbsp; Scores je Kopf: [B, h, T, d_h] @ [B, h, d_h, T] = [B, h, T, T]. Dann folgen Maske, Skalierung mit √head_dim (nicht √d_out), Softmax und Dropout.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        attn_scores <span style="color: #666">=</span> queries <span style="color: #666">@</span> keys<span style="color: #666">.</span>transpose(<span style="color: #666">2</span>, <span style="color: #666">3</span>)&#10;        mask_bool <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>mask<span style="color: #666">.</span>bool()[:num_tokens, :num_tokens]&#10;        attn_scores<span style="color: #666">.</span>masked_fill_(mask_bool, <span style="color: #666">-</span>torch<span style="color: #666">.</span>inf)&#10;        attn_weights <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(attn_scores <span style="color: #666">/</span> keys<span style="color: #666">.</span>shape[<span style="color: #666">-1</span>]<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;        attn_weights <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>dropout(attn_weights)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>⑥</b>&nbsp; Mischung je Kopf: [B, h, T, T] @ [B, h, T, d_h] = [B, h, T, d_h]. Dann kommen die Köpfe wieder nebeneinander: <code>transpose</code> führt zurück zu [B, T, h, d_h], <code>contiguous</code> legt den Speicher neu an, und <code>view</code> fügt zu [B, T, d_out] zusammen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        context_vec <span style="color: #666">=</span> (attn_weights <span style="color: #666">@</span> values)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)&#10;        context_vec <span style="color: #666">=</span> context_vec<span style="color: #666">.</span>contiguous()<span style="color: #666">.</span>view(b, num_tokens, <span style="color: #008000">self</span><span style="color: #666">.</span>d_out)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>⑦</b>&nbsp; Die Ausgabeprojektion lässt die Form [B, T, d_out] unverändert. <code>return_weights</code> ist eine Ergänzung für dieses Notebook, damit die Übungen auf die Gewichte der einzelnen Köpfe zugreifen können.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>        context_vec <span style="color: #666">=</span> <span style="color: #008000">self</span><span style="color: #666">.</span>out_proj(context_vec)&#10;        <span style="color: #008000; font-weight: bold">if</span> return_weights:&#10;            <span style="color: #008000; font-weight: bold">return</span> context_vec, attn_weights&#10;        <span style="color: #008000; font-weight: bold">return</span> context_vec&#10;&#10;&#10;torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">123</span>)&#10;mha <span style="color: #666">=</span> MultiHeadAttention(d_in<span style="color: #666">=3</span>, d_out<span style="color: #666">=2</span>, context_length<span style="color: #666">=6</span>, dropout<span style="color: #666">=0.0</span>, num_heads<span style="color: #666">=2</span>)&#10;kontext, gewichte_pro_kopf <span style="color: #666">=</span> mha(batch, return_weights<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Eingabe:"</span>, <span style="color: #008000">tuple</span>(batch<span style="color: #666">.</span>shape), <span style="color: #BA2121">"→ Ausgabe:"</span>, <span style="color: #008000">tuple</span>(kontext<span style="color: #666">.</span>shape), <span style="color: #BA2121">" Gewichte je Kopf:"</span>, <span style="color: #008000">tuple</span>(gewichte_pro_kopf<span style="color: #666">.</span>shape))&#10;<span style="color: #008000">print</span>()&#10;<span style="color: #008000; font-weight: bold">for</span> kopf <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">range</span>(<span style="color: #666">2</span>):&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">f"Kopf </span><span style="color: #A45A77; font-weight: bold">{</span>kopf<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">, Satz 0:"</span>)&#10;    zeige(gewichte_pro_kopf[<span style="color: #666">0</span>, kopf]<span style="color: #666">.</span>detach(), woerter, woerter)</code></pre></details>

In [ ]:
# Vorgabe: Multi-Head Attention — einfach ausführen
class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out muss durch num_heads teilbar sein"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x, return_weights=False):
        b, num_tokens, d_in = x.shape
        keys = self.W_key(x)          # [B, T, d_out]
        queries = self.W_query(x)
        values = self.W_value(x)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)

        keys = keys.transpose(1, 2)
        queries = queries.transpose(1, 2)
        values = values.transpose(1, 2)

        attn_scores = queries @ keys.transpose(2, 3)
        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        attn_scores.masked_fill_(mask_bool, -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)

        context_vec = (attn_weights @ values).transpose(1, 2)
        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        context_vec = self.out_proj(context_vec)
        if return_weights:
            return context_vec, attn_weights
        return context_vec


torch.manual_seed(123)
mha = MultiHeadAttention(d_in=3, d_out=2, context_length=6, dropout=0.0, num_heads=2)
kontext, gewichte_pro_kopf = mha(batch, return_weights=True)
print("Eingabe:", tuple(batch.shape), "→ Ausgabe:", tuple(kontext.shape), " Gewichte je Kopf:", tuple(gewichte_pro_kopf.shape))
print()
for kopf in range(2):
    print(f"Kopf {kopf}, Satz 0:")
    zeige(gewichte_pro_kopf[0, kopf].detach(), woerter, woerter)

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Diese Zelle zeigt nur, wie sich die Formen im <code>forward</code> ändern, Schritt für Schritt am Beispiel der Keys. Maske, Dropout und Ausgabeprojektion lässt sie weg, weil sie keine Form ändern. Deshalb stimmen nur die Formen mit dem echten <code>forward</code> überein, die Zahlen nicht.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>b, T, _ <span style="color: #666">=</span> batch<span style="color: #666">.</span>shape&#10;k <span style="color: #666">=</span> mha<span style="color: #666">.</span>W_key(batch);                                        <span style="color: #008000">print</span>(<span style="color: #BA2121">"W_key(x)        "</span>, <span style="color: #008000">tuple</span>(k<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, T, d_out]"</span>)&#10;k <span style="color: #666">=</span> k<span style="color: #666">.</span>view(b, T, mha<span style="color: #666">.</span>num_heads, mha<span style="color: #666">.</span>head_dim);               <span style="color: #008000">print</span>(<span style="color: #BA2121">"view            "</span>, <span style="color: #008000">tuple</span>(k<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, T, h, d_h]"</span>)&#10;k <span style="color: #666">=</span> k<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>);                                       <span style="color: #008000">print</span>(<span style="color: #BA2121">"transpose(1, 2) "</span>, <span style="color: #008000">tuple</span>(k<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, h, T, d_h]"</span>)&#10;s <span style="color: #666">=</span> mha<span style="color: #666">.</span>W_query(batch)<span style="color: #666">.</span>view(b, T, mha<span style="color: #666">.</span>num_heads, mha<span style="color: #666">.</span>head_dim)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>) <span style="color: #666">@</span> k<span style="color: #666">.</span>transpose(<span style="color: #666">2</span>, <span style="color: #666">3</span>)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Scores          "</span>, <span style="color: #008000">tuple</span>(s<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, h, T, T]"</span>)&#10;c <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(s <span style="color: #666">/</span> mha<span style="color: #666">.</span>head_dim<span style="color: #666">**0.5</span>, dim<span style="color: #666">=-1</span>) <span style="color: #666">@</span> mha<span style="color: #666">.</span>W_value(batch)<span style="color: #666">.</span>view(b, T, mha<span style="color: #666">.</span>num_heads, mha<span style="color: #666">.</span>head_dim)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)   <span style="color: #3D7B7B; font-style: italic"># ohne Maske, nur für die Form</span>&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Kontext je Kopf "</span>, <span style="color: #008000">tuple</span>(c<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, h, T, d_h]"</span>)&#10;c <span style="color: #666">=</span> c<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)<span style="color: #666">.</span>contiguous()<span style="color: #666">.</span>view(b, T, mha<span style="color: #666">.</span>d_out);    <span style="color: #008000">print</span>(<span style="color: #BA2121">"zusammengefügt  "</span>, <span style="color: #008000">tuple</span>(c<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, T, d_out]"</span>)</code></pre></details>

In [ ]:
# Vorgabe: die Formen auf dem Weg durch den Block — einfach ausführen
b, T, _ = batch.shape
k = mha.W_key(batch);                                        print("W_key(x)        ", tuple(k.shape), " [B, T, d_out]")
k = k.view(b, T, mha.num_heads, mha.head_dim);               print("view            ", tuple(k.shape), " [B, T, h, d_h]")
k = k.transpose(1, 2);                                       print("transpose(1, 2) ", tuple(k.shape), " [B, h, T, d_h]")
s = mha.W_query(batch).view(b, T, mha.num_heads, mha.head_dim).transpose(1, 2) @ k.transpose(2, 3)
print("Scores          ", tuple(s.shape), " [B, h, T, T]")
c = torch.softmax(s / mha.head_dim**0.5, dim=-1) @ mha.W_value(batch).view(b, T, mha.num_heads, mha.head_dim).transpose(1, 2)   # ohne Maske, nur für die Form
print("Kontext je Kopf ", tuple(c.shape), " [B, h, T, d_h]")
c = c.transpose(1, 2).contiguous().view(b, T, mha.d_out);    print("zusammengefügt  ", tuple(c.shape), " [B, T, d_out]")

<div style="border:2px solid #E97132; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; margin-right:-20px;">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#E97132; font-weight:bold; font-size:1.08em;">💭 Kurz überlegen</span></div>
<p>Die nächste Zelle verwendet wieder den DataLoader aus Lerneinheit 2. Sie holt einen Batch der Form <code>[8, 4]</code> aus *The Verdict* und schickt ihn durch Token- und Positions-Embedding und dann durch eine <code>MultiHeadAttention</code> in GPT-2-Größe. Legen Sie sich fest: <strong>Welche Form hat die Ausgabe?</strong> Und: Wie viele Parameter hat dieser eine Attention-Baustein?</p>

</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: ein echter Batch durch die Attention — einfach ausführen</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">os</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">urllib.request</span>&#10;&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">tiktoken</span>&#10;<span style="color: #008000; font-weight: bold">from</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">torch.utils.data</span><span style="color: #BBB"> </span><span style="color: #008000; font-weight: bold">import</span> Dataset, DataLoader&#10;&#10;&#10;KURS_REPO <span style="color: #666">=</span> <span style="color: #BA2121">"https://raw.githubusercontent.com/skuhswo/ann2/main/LE02/"</span>   <span style="color: #3D7B7B; font-style: italic"># Daten liegen bei Lerneinheit 2</span></code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; <code>the-verdict.txt</code> liegt bei Lerneinheit 2. Die Funktion sucht die Datei neben dem Notebook und im Ordner LE02 und lädt sie sonst aus dem Kurs-Repo.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">lade_text</span>(datei):&#10;    <span style="color: #008000; font-weight: bold">for</span> pfad <span style="color: #A2F; font-weight: bold">in</span> (datei, os<span style="color: #666">.</span>path<span style="color: #666">.</span>join(<span style="color: #BA2121">".."</span>, <span style="color: #BA2121">"LE02"</span>, datei)):&#10;        <span style="color: #008000; font-weight: bold">if</span> os<span style="color: #666">.</span>path<span style="color: #666">.</span>exists(pfad):&#10;            <span style="color: #008000; font-weight: bold">with</span> <span style="color: #008000">open</span>(pfad, <span style="color: #BA2121">"r"</span>, encoding<span style="color: #666">=</span><span style="color: #BA2121">"utf-8"</span>) <span style="color: #008000; font-weight: bold">as</span> f:&#10;                <span style="color: #008000; font-weight: bold">return</span> f<span style="color: #666">.</span>read()&#10;    urllib<span style="color: #666">.</span>request<span style="color: #666">.</span>urlretrieve(KURS_REPO <span style="color: #666">+</span> datei, datei)&#10;    <span style="color: #008000; font-weight: bold">with</span> <span style="color: #008000">open</span>(datei, <span style="color: #BA2121">"r"</span>, encoding<span style="color: #666">=</span><span style="color: #BA2121">"utf-8"</span>) <span style="color: #008000; font-weight: bold">as</span> f:&#10;        <span style="color: #008000; font-weight: bold">return</span> f<span style="color: #666">.</span>read()</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Dataset und DataLoader stammen aus Lerneinheit 2: ein Sliding Window über die Token-IDs, das Ziel ist die um eins verschobene Eingabe.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">class</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">GPTDatasetV1</span>(Dataset):&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">__init__</span>(<span style="color: #008000">self</span>, txt, tokenizer, max_length, stride):&#10;        <span style="color: #008000">self</span><span style="color: #666">.</span>input_ids, <span style="color: #008000">self</span><span style="color: #666">.</span>target_ids <span style="color: #666">=</span> [], []&#10;        token_ids <span style="color: #666">=</span> tokenizer<span style="color: #666">.</span>encode(txt, allowed_special<span style="color: #666">=</span>{<span style="color: #BA2121">"&lt;|endoftext|&gt;"</span>})&#10;        <span style="color: #008000; font-weight: bold">for</span> i <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">range</span>(<span style="color: #666">0</span>, <span style="color: #008000">len</span>(token_ids) <span style="color: #666">-</span> max_length, stride):&#10;            <span style="color: #008000">self</span><span style="color: #666">.</span>input_ids<span style="color: #666">.</span>append(torch<span style="color: #666">.</span>tensor(token_ids[i:i <span style="color: #666">+</span> max_length]))&#10;            <span style="color: #008000">self</span><span style="color: #666">.</span>target_ids<span style="color: #666">.</span>append(torch<span style="color: #666">.</span>tensor(token_ids[i <span style="color: #666">+</span> <span style="color: #666">1</span>:i <span style="color: #666">+</span> max_length <span style="color: #666">+</span> <span style="color: #666">1</span>]))&#10;&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">__len__</span>(<span style="color: #008000">self</span>):&#10;        <span style="color: #008000; font-weight: bold">return</span> <span style="color: #008000">len</span>(<span style="color: #008000">self</span><span style="color: #666">.</span>input_ids)&#10;&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">__getitem__</span>(<span style="color: #008000">self</span>, idx):&#10;        <span style="color: #008000; font-weight: bold">return</span> <span style="color: #008000">self</span><span style="color: #666">.</span>input_ids[idx], <span style="color: #008000">self</span><span style="color: #666">.</span>target_ids[idx]&#10;&#10;&#10;raw_text <span style="color: #666">=</span> lade_text(<span style="color: #BA2121">"the-verdict.txt"</span>)&#10;tokenizer <span style="color: #666">=</span> tiktoken<span style="color: #666">.</span>get_encoding(<span style="color: #BA2121">"gpt2"</span>)&#10;dataloader <span style="color: #666">=</span> DataLoader(GPTDatasetV1(raw_text, tokenizer, max_length<span style="color: #666">=4</span>, stride<span style="color: #666">=4</span>), batch_size<span style="color: #666">=8</span>, shuffle<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">False</span>)&#10;eingabe_ids, ziel_ids <span style="color: #666">=</span> <span style="color: #008000">next</span>(<span style="color: #008000">iter</span>(dataloader))&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Token-IDs:        "</span>, <span style="color: #008000">tuple</span>(eingabe_ids<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, T]"</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Die beiden Embedding-Tabellen aus Lerneinheit 2 haben GPT-2-Größe: 50.257 Tokens und 1.024 Positionen, je 768 Dimensionen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>vocab_size, context_length, d_model <span style="color: #666">=</span> <span style="color: #666">50_257</span>, <span style="color: #666">1_024</span>, <span style="color: #666">768</span>&#10;torch<span style="color: #666">.</span>manual_seed(<span style="color: #666">123</span>)&#10;token_embedding <span style="color: #666">=</span> nn<span style="color: #666">.</span>Embedding(vocab_size, d_model)&#10;pos_embedding <span style="color: #666">=</span> nn<span style="color: #666">.</span>Embedding(context_length, d_model)&#10;x <span style="color: #666">=</span> token_embedding(eingabe_ids) <span style="color: #666">+</span> pos_embedding(torch<span style="color: #666">.</span>arange(eingabe_ids<span style="color: #666">.</span>shape[<span style="color: #666">1</span>]))&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Eingabe-Tensor:   "</span>, <span style="color: #008000">tuple</span>(x<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, T, d]"</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>④</b>&nbsp; Die Attention in GPT-2-Größe hat 12 Köpfe à 64 Dimensionen, Dropout 0,1 und Bias wie in GPT-2. Die Form bleibt erhalten, aber jeder Vektor enthält jetzt Information aus seinem Kontext.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>mha_gpt2 <span style="color: #666">=</span> MultiHeadAttention(d_model, d_model, context_length, dropout<span style="color: #666">=0.1</span>, num_heads<span style="color: #666">=12</span>, qkv_bias<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>)&#10;ausgabe <span style="color: #666">=</span> mha_gpt2(x)&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">"Nach der Attention:"</span>, <span style="color: #008000">tuple</span>(ausgabe<span style="color: #666">.</span>shape), <span style="color: #BA2121">" [B, T, d]"</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>⑤</b>&nbsp; Parameter zählen: vier Projektionen à 768 × 768 plus vier Bias-Vektoren.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>parameter <span style="color: #666">=</span> <span style="color: #008000">sum</span>(p<span style="color: #666">.</span>numel() <span style="color: #008000; font-weight: bold">for</span> p <span style="color: #A2F; font-weight: bold">in</span> mha_gpt2<span style="color: #666">.</span>parameters())&#10;<span style="color: #008000">print</span>(<span style="color: #BA2121">f"Parameter des Bausteins: </span><span style="color: #A45A77; font-weight: bold">{</span>parameter<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">,</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> (4 · (768 · 768 + 768) = </span><span style="color: #A45A77; font-weight: bold">{</span><span style="color: #666">4</span><span style="color: #BBB"> </span><span style="color: #666">*</span><span style="color: #BBB"> </span>(<span style="color: #666">768</span><span style="color: #BBB"> </span><span style="color: #666">*</span><span style="color: #BBB"> </span><span style="color: #666">768</span><span style="color: #BBB"> </span><span style="color: #666">+</span><span style="color: #BBB"> </span><span style="color: #666">768</span>)<span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">,</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">)"</span>)</code></pre></details>

In [ ]:
# Vorgabe: ein echter Batch durch die Attention — einfach ausführen
import os
import urllib.request

import tiktoken
from torch.utils.data import Dataset, DataLoader


KURS_REPO = "https://raw.githubusercontent.com/skuhswo/ann2/main/LE02/"   # Daten liegen bei Lerneinheit 2


def lade_text(datei):
    for pfad in (datei, os.path.join("..", "LE02", datei)):
        if os.path.exists(pfad):
            with open(pfad, "r", encoding="utf-8") as f:
                return f.read()
    urllib.request.urlretrieve(KURS_REPO + datei, datei)
    with open(datei, "r", encoding="utf-8") as f:
        return f.read()


class GPTDatasetV1(Dataset):
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids, self.target_ids = [], []
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        for i in range(0, len(token_ids) - max_length, stride):
            self.input_ids.append(torch.tensor(token_ids[i:i + max_length]))
            self.target_ids.append(torch.tensor(token_ids[i + 1:i + max_length + 1]))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return self.input_ids[idx], self.target_ids[idx]


raw_text = lade_text("the-verdict.txt")
tokenizer = tiktoken.get_encoding("gpt2")
dataloader = DataLoader(GPTDatasetV1(raw_text, tokenizer, max_length=4, stride=4), batch_size=8, shuffle=False)
eingabe_ids, ziel_ids = next(iter(dataloader))
print("Token-IDs:        ", tuple(eingabe_ids.shape), " [B, T]")

vocab_size, context_length, d_model = 50_257, 1_024, 768
torch.manual_seed(123)
token_embedding = nn.Embedding(vocab_size, d_model)
pos_embedding = nn.Embedding(context_length, d_model)
x = token_embedding(eingabe_ids) + pos_embedding(torch.arange(eingabe_ids.shape[1]))
print("Eingabe-Tensor:   ", tuple(x.shape), " [B, T, d]")

mha_gpt2 = MultiHeadAttention(d_model, d_model, context_length, dropout=0.1, num_heads=12, qkv_bias=True)
ausgabe = mha_gpt2(x)
print("Nach der Attention:", tuple(ausgabe.shape), " [B, T, d]")

parameter = sum(p.numel() for p in mha_gpt2.parameters())
print(f"Parameter des Bausteins: {parameter:,} (4 · (768 · 768 + 768) = {4 * (768 * 768 + 768):,})")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E75B6; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Formen und Zahlen zum Mitnehmen</div>
<div style="background:#EAF2FA; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Ein Attention-Baustein nimmt <code>[B, T, d_in]</code> entgegen und gibt <code>[B, T, d_out]</code> zurück. In GPT-2 sind beide 768, deshalb bleibt die Form dort erhalten (das kleine Beispiel oben hat 3 und 2). Dazwischen entstehen die Gewichte der Form <code>[B, h, T, T]</code>. Ihre Größe wächst quadratisch mit der Folgenlänge: Bei T = 1.024 sind das rund eine Million Einträge je Kopf, in float32 also 48 MiB je Block und Sequenz.</p>
<p>GPT-2 small hat je Block 2.362.368 Attention-Parameter, in 12 Blöcken 28,3 Mio., rund 23 % des Modells. Ohne die Bias-Vektoren von Query, Key und Value (so ist die Klasse voreingestellt, und so rechnet auch die Buchlösung zu Übung 3.3) sind es 2.360.064.</p>

</div>
</div>

### <span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:1px solid rgba(21,96,130,0.45); padding-bottom:2px;  padding-top:16px;">3.4.1&nbsp;&nbsp;Übungen</span>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#B4532A; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">15 min</span>🔧 Aufgabe 3.4 #1 · Bauen · in der Vorlesung</div>
<div style="background:#FBEDE6; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie sehen, wie Köpfe durch ihre Projektionen definiert sind und was sich ändert, wenn Keys und Values geteilt werden. Diese Keys und Values werden bei der Textgenerierung im KV-Cache gespeichert. Grouped-Query Attention (Projektthema I8) liegt zwischen Multi-Head und Multi-Query Attention.</em></p>
<p>Bauen Sie aus <code>MultiHeadAttention</code> eine <strong>Multi-Query Attention</strong> (Shazeer 2019): Alle Köpfe teilen sich einen Key und einen Value. <code>W_key</code> und <code>W_value</code> sind deshalb nur noch <code>head_dim</code> breit (768 × 64 statt 768 × 768), ihre Ausgabe nutzen alle Köpfe gemeinsam. Konstruktor und der Rest des <code>forward</code> sind vorgegeben. Sie füllen drei Lücken:</p>
<ol>
<li>Keys in die Form <code>[B, 1, T, head_dim]</code> bringen (ein Kopf).</li>
<li>Keys per <code>expand</code> auf <code>[B, num_heads, T, head_dim]</code> verteilen. <code>expand</code> legt keine Kopie an, sondern wiederholt nur die Sicht auf denselben Speicher.</li>
<li>Dasselbe für die Values.</li>
</ol>
<p>Die Prüfzelle testet Form, Kausalität und Parameterzahl, und sie vergleicht mit einer <code>MultiHeadAttention</code>, in der alle Köpfe dieselben Key- und Value-Matrizen bekommen haben.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#B4532A; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #B4532A55"><p><strong>Hinweis:</strong> <code>self.W_key(x).view(b, num_tokens, 1, self.head_dim).transpose(1, 2)</code> ergibt <code>[B, 1, T, head_dim]</code>, danach <code>.expand(b, self.num_heads, num_tokens, self.head_dim)</code>.</p></div></details>

</div>
</div>

In [ ]:
class MultiQueryAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        assert d_out % num_heads == 0, "d_out muss durch num_heads teilbar sein"
        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key = nn.Linear(d_in, self.head_dim, bias=qkv_bias)      # nur ein Kopf breit
        self.W_value = nn.Linear(d_in, self.head_dim, bias=qkv_bias)    # nur ein Kopf breit
        self.out_proj = nn.Linear(d_out, d_out)
        self.dropout = nn.Dropout(dropout)
        self.register_buffer("mask", torch.triu(torch.ones(context_length, context_length), diagonal=1))

    def forward(self, x):
        b, num_tokens, d_in = x.shape
        queries = self.W_query(x).view(b, num_tokens, self.num_heads, self.head_dim).transpose(1, 2)   # [B, h, T, d_h]

        # TODO Lücke 1: keys in die Form [B, 1, T, d_h]
        keys = ...
        # TODO Lücke 2: keys auf alle Köpfe verteilen, Form [B, h, T, d_h]
        keys = ...
        # TODO Lücke 3: dasselbe für values
        values = ...

        attn_scores = queries @ keys.transpose(2, 3)
        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        attn_weights = torch.softmax(attn_scores / keys.shape[-1]**0.5, dim=-1)
        attn_weights = self.dropout(attn_weights)
        context_vec = (attn_weights @ values).transpose(1, 2).contiguous().view(b, num_tokens, self.d_out)
        return self.out_proj(context_vec)

In [ ]:
# Prüfzelle — einfach ausführen
torch.manual_seed(0)
mqa = MultiQueryAttention(768, 768, 1024, dropout=0.0, num_heads=12, qkv_bias=True)
mqa.eval()
_n = sum(p.numel() for p in mqa.parameters())
assert _n == 1_279_616, f"Parameterzahl sollte 1.279.616 sein, ist {_n:,}"
_x = torch.randn(2, 5, 768)
with torch.no_grad():
    _out = mqa(_x)
    assert _out.shape == (2, 5, 768), f"Ausgabeform sollte (2, 5, 768) sein, ist {tuple(_out.shape)}"
    _x2 = _x.clone(); _x2[:, 4] += 1.0
    assert torch.allclose(mqa(_x2)[:, :4], _out[:, :4], atol=1e-5), "Leck: eine Änderung an Position 4 verändert frühere Ausgaben"
    # Vergleich: MultiHeadAttention, in der alle 12 Köpfe dieselben Key- und Value-Matrizen bekommen
    _ref = MultiHeadAttention(768, 768, 1024, dropout=0.0, num_heads=12, qkv_bias=True)
    _ref.eval()
    _ref.W_query.load_state_dict(mqa.W_query.state_dict())
    _ref.out_proj.load_state_dict(mqa.out_proj.state_dict())
    _ref.W_key.weight.copy_(mqa.W_key.weight.repeat(12, 1)); _ref.W_key.bias.copy_(mqa.W_key.bias.repeat(12))
    _ref.W_value.weight.copy_(mqa.W_value.weight.repeat(12, 1)); _ref.W_value.bias.copy_(mqa.W_value.bias.repeat(12))
    assert torch.allclose(_ref(_x), _out, atol=1e-4), "Mit geteilten Keys und Values muss die Multi-Head Attention dieselbe Ausgabe liefern. Prüfen Sie view und expand."
print(f"Parameter: {_n:,} statt {sum(p.numel() for p in mha_gpt2.parameters()):,}")
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>🔬 Aufgabe 3.4 #2 · Experiment · Hausaufgabe, nach Wahl</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: <code>d_out</code> bedeutet beim Wrapper etwas anderes als bei der fertigen Klasse.</em></p>
<p>Die Aufgabe greift Raschkas Übung 3.2 auf. Der <code>MultiHeadAttentionWrapper</code> ist eine Liste einzelner Köpfe, deren Ausgaben nebeneinandergelegt werden. Anders als im Buch, wo der Wrapper aus <code>CausalAttention</code> besteht, baut ihn die Zelle aus <code>MultiHeadAttention</code> mit je einem Kopf. Dadurch hat jeder Kopf eine eigene Ausgabeprojektion, und es sind 22 statt 18 Parameter. An der Form der Ausgabe ändert das nichts. Sagen Sie die Ausgabeform für zwei Köpfe mit <code>d_out = 1</code> vorher (<code>vorhersage_form</code>), und ob <code>MultiHeadAttention(3, 1, 6, 0.0, num_heads=2)</code> läuft (<code>vorhersage_laeuft</code>).</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Beim Wrapper ist <code>d_out</code> die Breite je Kopf, bei der fertigen Klasse die Gesamtbreite.</p></div></details>

</div>
</div>

In [ ]:
class MultiHeadAttentionWrapper(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()
        self.heads = nn.ModuleList([MultiHeadAttention(d_in, d_out, context_length, dropout, 1, qkv_bias) for _ in range(num_heads)])

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)


vorhersage_form = (..., ..., ...)
vorhersage_laeuft = ...              # True oder False

wrapper = MultiHeadAttentionWrapper(3, 1, 6, 0.0, num_heads=2)
form_wrapper = tuple(wrapper(batch).shape)
try:
    MultiHeadAttention(3, 1, 6, 0.0, num_heads=2)
    fehler = "kein Fehler"
except AssertionError as e:
    fehler = str(e)
print(form_wrapper, "|", fehler)

In [ ]:
# Prüfzelle — einfach ausführen
assert isinstance(vorhersage_form, tuple) and all(isinstance(v, int) for v in vorhersage_form) and isinstance(vorhersage_laeuft, bool), "Tragen Sie beide Vorhersagen ein"
assert form_wrapper == (2, 6, 2) and "teilbar" in fehler
print("Beobachtet:", form_wrapper, "und:", fehler, "— Ihre Vorhersagen:", vorhersage_form, vorhersage_laeuft)
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">10 min</span>🔬 Aufgabe 3.4 #3 · Experiment · in der Vorlesung</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie sehen, wo die Parameter der Attention sitzen und wie Parameterzahl und Aktivierungsspeicher jeweils von T abhängen.</em></p>
<p>Die Aufgabe verbindet Raschkas Übung 3.3 mit einer Speicherrechnung. Die Zelle zählt die Parameter zweier Module in GPT-2-Größe, einmal ohne und einmal mit Q/K/V-Bias, und berechnet den Speicher der Gewichtsmatrix <code>[B, h, T, T]</code>.</p>
<p>Tragen Sie vorab zwei Zahlen ein. In <code>vorhersage_differenz</code>: Um wie viele Parameter unterscheiden sich die beiden Module, und warum? In <code>vorhersage_mib</code>: Wie viel Speicher belegt die Gewichtsmatrix für B = 8, h = 12, T = 1.024 in float32, in MiB? Rechnen Sie beides von Hand, bevor Sie ausführen. Dann ändern Sie <code>T</code> auf 2.048 und sehen nach, was mit Parametern und Speicher passiert.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> Ein float32 belegt 4 Bytes, 1 MiB sind 2**20 Bytes. Die Ausgabeprojektion hat immer einen Bias.</p></div></details>

</div>
</div>

In [ ]:
vorhersage_differenz = ...     # Parameter mit Bias minus Parameter ohne Bias
vorhersage_mib = ...           # MiB für [8, 12, 1024, 1024] in float32
T = 1024                       # danach 2048 probieren

params_ohne = sum(p.numel() for p in MultiHeadAttention(768, 768, T, 0.1, 12, qkv_bias=False).parameters())
params_mit = sum(p.numel() for p in MultiHeadAttention(768, 768, T, 0.1, 12, qkv_bias=True).parameters())
speicher_mib = 8 * 12 * T * T * 4 / 2**20
klein = torch.zeros(8, 12, 16, 16)
print(f"T = {T}: Parameter ohne Bias {params_ohne:,}, mit Bias {params_mit:,}, Differenz {params_mit - params_ohne:,}")
print(f"Gewichtsmatrix [8, 12, {T}, {T}] in float32: {speicher_mib:.0f} MiB (Formel geprüft an [8, 12, 16, 16]: {klein.element_size() * klein.nelement()} Bytes = {8 * 12 * 16 * 16 * 4})")

In [ ]:
# Prüfzelle — einfach ausführen
assert isinstance(vorhersage_differenz, int) and isinstance(vorhersage_mib, (int, float)), "Tragen Sie beide Vorhersagen als Zahlen ein"
assert params_ohne == 2_360_064 and params_mit == 2_362_368, "Erwartet: 2.360.064 ohne und 2.362.368 mit Q/K/V-Bias"
print(f"Beobachtet: Differenz {params_mit - params_ohne:,}, Speicher bei T = 1024: 384 MiB — Ihre Vorhersagen: {vorhersage_differenz:,}, {vorhersage_mib}")
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#2E7D32; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">8 min</span>🔬 Aufgabe 3.4 #4 · Experiment · Hausaufgabe, nach Wahl</div>
<div style="background:#EAF4EB; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie verfolgen die Formen durch die ganze Verarbeitungskette und sehen, was die Gewichte eines untrainierten Modells aussagen und was nicht.</em></p>
<p>Die Zelle schickt das Laufbeispiel „Übermorgen fährt das Rheinschiff nach Worms.“ durch die ganze Kette: <code>tokenizer</code> (tiktoken, GPT-2), <code>token_embedding</code> plus <code>pos_embedding</code>, dann <code>mha_gpt2</code> mit <code>return_weights=True</code>, und liest aus den Gewichten von Kopf 0 die Zeile für das Token „ Worm“ (mit führendem Leerzeichen, so zerlegt der GPT-2-Tokenizer das Wort).</p>
<p>Vorhersagen vorab: die drei Formen (<code>vorhersage_formen</code>) und ab welchem Index die Zeile für „ Worm“ sicher null ist, egal wie die Gewichte initialisiert sind (<code>vorhersage_null_ab</code>). Dann ändern Sie <code>kopf</code> und vergleichen die Zeilen.</p>
<details style="margin-top:10px"><summary style="cursor:pointer; color:#2E7D32; font-weight:600; font-size:0.9em">Hinweis anzeigen</summary><div style="margin-top:8px; padding-top:8px; border-top:1px dashed #2E7D3255"><p><strong>Hinweis:</strong> 22 Tokens, „ Worm“ ist das Token mit Index 19. Die Position von „ Worm“ findet die Zelle mit <code>ids.index(...)</code>.</p></div></details>

</div>
</div>

In [ ]:
vorhersage_formen = [(..., ...), (..., ..., ...), (..., ..., ...)]   # Tokens, Embedding, Ausgabe
vorhersage_null_ab = ...                                            # Index, ab dem die Zeile null ist
kopf = 0

satz = "Übermorgen fährt das Rheinschiff nach Worms."
mha_gpt2.eval()
ids = tokenizer.encode(satz)
id_tensor = torch.tensor([ids])
with torch.no_grad():
    x_satz = token_embedding(id_tensor) + pos_embedding(torch.arange(len(ids)))
    ausgabe_satz, gewichte_satz = mha_gpt2(x_satz, return_weights=True)
formen = [tuple(id_tensor.shape), tuple(x_satz.shape), tuple(ausgabe_satz.shape)]
position_worm = ids.index(tokenizer.encode(" Worm")[0])
gewichte_worm = gewichte_satz[0, kopf, position_worm]
print("Formen:", formen, " Position von Worm:", position_worm)
print(f"Zeile für Worm, Kopf {kopf}:", gewichte_worm)

In [ ]:
# Prüfzelle — einfach ausführen
assert isinstance(vorhersage_null_ab, int) and all(isinstance(f, tuple) for f in vorhersage_formen), "Tragen Sie beide Vorhersagen ein"
assert formen == [(1, 22), (1, 22, 768), (1, 22, 768)]
assert float(gewichte_worm[position_worm + 1:].abs().sum()) == 0 and abs(float(gewichte_worm.sum()) - 1) < 1e-4
print(f"Beobachtet: Formen {formen}, null ab Index {position_worm + 1} — Ihre Vorhersagen: {vorhersage_formen}, {vorhersage_null_ab}")
print("Alle Prüfungen bestanden ✓")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#7E3FA0; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right; font-weight:600; font-size:0.85em; opacity:0.95;">5 min</span>💭 Aufgabe 3.4 #5 · Denkfrage · Hausaufgabe, nach Wahl</div>
<div style="background:#F3EAF8; color:#1a1a1a; padding:12px 16px; margin:0;">
<p style="margin-top:0; color:#475569;"><em>Worum es geht: Sie üben, eine Behauptung über Attention-Gewichte zu prüfen, bevor Sie sie als Erklärung nehmen.</em></p>
<p>Sehen Sie sich Abbildung 4 aus Vaswani et al. 2017 noch einmal an (in den Folien der Forschungsfall): zwei Köpfe, die bei „its“ scharf auf „Law“ und auf „application“ zeigen. Schreiben Sie drei Sätze: Was zeigt die Abbildung? Was zeigt sie nicht? Was müsste man prüfen, bevor man sagt, der Kopf „löst Pronomen auf“?</p>

</div>
</div>

In [ ]:
antwort = """
1. Die Abbildung zeigt: ...
2. Sie zeigt nicht: ...
3. Zu prüfen wäre: ...
"""

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 3.4 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Warum hat jeder Kopf <code>d_out / num_heads</code> Dimensionen und nicht <code>d_out</code>?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Warum werden die Kontextvektoren der zwölf Köpfe nebeneinandergelegt (der letzte <code>view</code> auf <code>d_out</code>) und nicht addiert, etwa mit einem gelernten Gewicht je Kopf?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Wozu braucht es <code>contiguous()</code> vor dem letzten <code>view</code>?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">4.</b> Um welchen Faktor schrumpft der KV-Cache bei Multi-Query Attention gegenüber Multi-Head Attention mit 12 Köpfen, und was bleibt gleich groß?</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#107C72; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right"><span style="border:1px solid rgba(255,255,255,0.85); color:#fff; font-size:0.72em; font-weight:bold; padding:1px 9px; border-radius:10px; margin-left:8px; vertical-align:1px;">NICHT PRÜFUNGSRELEVANT</span></span>Vertiefung: Attention-Köpfe ansehen, und was sie tun</div>
<div style="background:#E7F4F2; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Mit <a href="https://github.com/jessevig/bertviz">BertViz</a> (Vig 2019) lassen sich die Gewichte aller Köpfe eines trainierten Modells ansehen, etwa für GPT-2 small und den Satz aus dem Einstieg der Folien. Der einfachste Einstieg ist das im README verlinkte Colab-Notebook, in dem Modell und Visualisierung schon geladen sind. Achten Sie darauf, wie viele Köpfe den Großteil ihres Gewichts auf das erste Token legen. Das ist das Muster, das Xiao et al. 2024 als Attention Sink beschreiben (Projektthema I9).</p>
<p><strong>Was Köpfe tun.</strong> Olsson et al. 2022, *In-context Learning and Induction Heads*: Köpfe, die das Muster [A][B] … [A] → [B] fortsetzen. Wang et al. 2022, *Interpretability in the Wild*: ein Schaltkreis aus 26 Köpfen in GPT-2 small für eine einzige Aufgabe. McDougall et al. 2023, *Copy Suppression*: ein Kopf, der auf ein früheres Token achtet, um genau dessen Vorhersage zu unterdrücken. Ein hohes Gewicht bedeutet also nicht einmal, dass ein Token positiv zur Vorhersage beiträgt. Yin und Steinhardt 2025, *Which Attention Heads Matter for In-Context Learning?* (ICML): Ablationen in zwölf Modellen zeigen, dass das Few-Shot-In-Context-Learning vor allem an Function-Vector-Köpfen hängt und weniger an Induction Heads, besonders in größeren Modellen.</p>

</div>
</div>

## <a id="kap-attention-in-gro-en-llms"></a><span class="zwischentitel" style="display:block; color:#0E2841; border-bottom:3px solid #156082; padding-bottom:4px;  padding-top:26px;">3.5&nbsp;&nbsp;Attention in großen LLMs</span>

Dieses Kapitel hat keine Übungen und ist nicht prüfungsrelevant. Es ordnet ein, wie die Attention aus Kapitel 3.4 in heutigen Modellen eingesetzt wird: als fertiger Kern in PyTorch, mit einem KV-Cache bei der Textgenerierung und mit den Varianten, die aktuelle Modelle an der Rechnung ändern. Die Folien gehen die Verfahren einzeln durch, hier stehen Code, Zahlen und die Leseliste. Die Projektthemen I5, I8 und I9 setzen hier an.

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#107C72; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right"><span style="border:1px solid rgba(255,255,255,0.85); color:#fff; font-size:0.72em; font-weight:bold; padding:1px 9px; border-radius:10px; margin-left:8px; vertical-align:1px;">NICHT PRÜFUNGSRELEVANT</span></span>Vertiefung: Die eingebaute Attention von PyTorch</div>
<div style="background:#E7F4F2; color:#1a1a1a; padding:12px 16px; margin:0;">
<p><code>torch.nn.functional.scaled_dot_product_attention</code> berechnet den Kern der Attention (Scores, Maske, Softmax, Mischung) in einem einzigen Aufruf. Je nach Eingabe und Hardware wählt die Funktion einen effizienten Kern, zum Beispiel FlashAttention (Dao et al. 2022). FlashAttention berechnet dieselbe Attention exakt, also ohne Näherung (bitgleich müssen die Gleitkommaergebnisse nicht sein), schreibt die Gewichtsmatrix <code>[B, h, T, T]</code> aber nie vollständig in den Speicher. Der Rechenaufwand bleibt quadratisch in T, gespart werden Speicherzugriffe. Das Problem langer Kontexte löst FlashAttention damit nicht, es macht die dichte Attention nur schneller und sparsamer. Die Projektionen und die Ausgabeprojektion muss man weiterhin selbst berechnen. Die Zelle vergleicht die Funktion mit dem Kern unserer Klasse. Beide erhalten dieselben Q, K und V, Dropout ist abgeschaltet.</p>

</div>
</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">time</span>&#10;<span style="color: #008000; font-weight: bold">import</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">torch.nn.functional</span><span style="color: #BBB"> </span><span style="color: #008000; font-weight: bold">as</span><span style="color: #BBB"> </span><span style="color: #00F; font-weight: bold">F</span>&#10;&#10;mha_gpt2<span style="color: #666">.</span>eval()&#10;<span style="color: #008000; font-weight: bold">with</span> torch<span style="color: #666">.</span>no_grad():&#10;    b_, T_, _ <span style="color: #666">=</span> x<span style="color: #666">.</span>shape&#10;    q <span style="color: #666">=</span> mha_gpt2<span style="color: #666">.</span>W_query(x)<span style="color: #666">.</span>view(b_, T_, <span style="color: #666">12</span>, <span style="color: #666">64</span>)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)&#10;    k <span style="color: #666">=</span> mha_gpt2<span style="color: #666">.</span>W_key(x)<span style="color: #666">.</span>view(b_, T_, <span style="color: #666">12</span>, <span style="color: #666">64</span>)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)&#10;    v <span style="color: #666">=</span> mha_gpt2<span style="color: #666">.</span>W_value(x)<span style="color: #666">.</span>view(b_, T_, <span style="color: #666">12</span>, <span style="color: #666">64</span>)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Unser eigener Kern rechnet Scores, Maske, Softmax und Mischung wie im <code>forward</code> von <code>MultiHeadAttention</code>.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    s <span style="color: #666">=</span> (q <span style="color: #666">@</span> k<span style="color: #666">.</span>transpose(<span style="color: #666">2</span>, <span style="color: #666">3</span>))<span style="color: #666">.</span>masked_fill(mha_gpt2<span style="color: #666">.</span>mask<span style="color: #666">.</span>bool()[:T_, :T_], <span style="color: #666">-</span>torch<span style="color: #666">.</span>inf)&#10;    eigener_kern <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(s <span style="color: #666">/</span> <span style="color: #666">64**0.5</span>, dim<span style="color: #666">=-1</span>) <span style="color: #666">@</span> v</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Derselbe Kern aus PyTorch: <code>is_causal=True</code> setzt die Causal Mask intern. <code>dropout_p=0.0</code> steht ausdrücklich da, denn die Funktion kennt <code>model.eval()</code> nicht, ihr Dropout wird nur über diesen Parameter gesteuert (Voreinstellung ebenfalls 0).</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    pytorch_kern <span style="color: #666">=</span> F<span style="color: #666">.</span>scaled_dot_product_attention(q, k, v, is_causal<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>, dropout_p<span style="color: #666">=0.0</span>)&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">"gleich:"</span>, torch<span style="color: #666">.</span>allclose(eigener_kern, pytorch_kern, atol<span style="color: #666">=1e-5</span>))</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Ein grober Zeitvergleich an einer längeren Folge, bewusst auf der CPU: Die Tensoren werden ohne Geräteangabe angelegt und liegen deshalb auch dann im Hauptspeicher, wenn Colab eine GPU hat. Die Zelle baut die Maske einmal, macht einen Aufwärmlauf und misst dann zehn Durchläufe. Das ist eine Illustration, kein Benchmark: Welchen Kern PyTorch wählt, hängt von Eingabe und Hardware ab, und auf der GPU bräuchte die Zeitmessung zusätzlich eine Synchronisation.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    q, k, v <span style="color: #666">=</span> (torch<span style="color: #666">.</span>randn(<span style="color: #666">1</span>, <span style="color: #666">12</span>, <span style="color: #666">1024</span>, <span style="color: #666">64</span>) <span style="color: #008000; font-weight: bold">for</span> _ <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">range</span>(<span style="color: #666">3</span>))&#10;    maske <span style="color: #666">=</span> torch<span style="color: #666">.</span>triu(torch<span style="color: #666">.</span>ones(<span style="color: #666">1024</span>, <span style="color: #666">1024</span>), diagonal<span style="color: #666">=1</span>)<span style="color: #666">.</span>bool()&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">eigener</span>():&#10;        s <span style="color: #666">=</span> (q <span style="color: #666">@</span> k<span style="color: #666">.</span>transpose(<span style="color: #666">2</span>, <span style="color: #666">3</span>))<span style="color: #666">.</span>masked_fill(maske, <span style="color: #666">-</span>torch<span style="color: #666">.</span>inf)&#10;        <span style="color: #008000; font-weight: bold">return</span> torch<span style="color: #666">.</span>softmax(s <span style="color: #666">/</span> <span style="color: #666">8</span>, dim<span style="color: #666">=-1</span>) <span style="color: #666">@</span> v&#10;    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">pytorch</span>():&#10;        <span style="color: #008000; font-weight: bold">return</span> F<span style="color: #666">.</span>scaled_dot_product_attention(q, k, v, is_causal<span style="color: #666">=</span><span style="color: #008000; font-weight: bold">True</span>, dropout_p<span style="color: #666">=0.0</span>)&#10;    <span style="color: #008000; font-weight: bold">for</span> f <span style="color: #A2F; font-weight: bold">in</span> (eigener, pytorch):&#10;        f()                                   <span style="color: #3D7B7B; font-style: italic"># Aufwärmen</span>&#10;        t0 <span style="color: #666">=</span> time<span style="color: #666">.</span>perf_counter()&#10;        <span style="color: #008000; font-weight: bold">for</span> _ <span style="color: #A2F; font-weight: bold">in</span> <span style="color: #008000">range</span>(<span style="color: #666">10</span>):&#10;            f()&#10;        <span style="color: #008000">print</span>(<span style="color: #BA2121">f"</span><span style="color: #A45A77; font-weight: bold">{</span>f<span style="color: #666">.</span><span style="color: #19177C">__name__</span><span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">8s</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> </span><span style="color: #A45A77; font-weight: bold">{</span>(time<span style="color: #666">.</span>perf_counter()<span style="color: #BBB"> </span><span style="color: #666">-</span><span style="color: #BBB"> </span>t0)<span style="color: #BBB"> </span><span style="color: #666">/</span><span style="color: #BBB"> </span><span style="color: #666">10</span><span style="color: #BBB"> </span><span style="color: #666">*</span><span style="color: #BBB"> </span><span style="color: #666">1000</span><span style="color: #A45A77; font-weight: bold">:</span><span style="color: #BA2121">6.1f</span><span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121"> ms je Durchlauf (T = 1024, 12 Köpfe, Gerät: </span><span style="color: #A45A77; font-weight: bold">{</span>q<span style="color: #666">.</span>device<span style="color: #A45A77; font-weight: bold">}</span><span style="color: #BA2121">)"</span>)</code></pre></details>

In [ ]:
import time
import torch.nn.functional as F

mha_gpt2.eval()
with torch.no_grad():
    b_, T_, _ = x.shape
    q = mha_gpt2.W_query(x).view(b_, T_, 12, 64).transpose(1, 2)
    k = mha_gpt2.W_key(x).view(b_, T_, 12, 64).transpose(1, 2)
    v = mha_gpt2.W_value(x).view(b_, T_, 12, 64).transpose(1, 2)

    s = (q @ k.transpose(2, 3)).masked_fill(mha_gpt2.mask.bool()[:T_, :T_], -torch.inf)
    eigener_kern = torch.softmax(s / 64**0.5, dim=-1) @ v
    pytorch_kern = F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
    print("gleich:", torch.allclose(eigener_kern, pytorch_kern, atol=1e-5))

    q, k, v = (torch.randn(1, 12, 1024, 64) for _ in range(3))
    maske = torch.triu(torch.ones(1024, 1024), diagonal=1).bool()
    def eigener():
        s = (q @ k.transpose(2, 3)).masked_fill(maske, -torch.inf)
        return torch.softmax(s / 8, dim=-1) @ v
    def pytorch():
        return F.scaled_dot_product_attention(q, k, v, is_causal=True, dropout_p=0.0)
    for f in (eigener, pytorch):
        f()                                   # Aufwärmen
        t0 = time.perf_counter()
        for _ in range(10):
            f()
        print(f"{f.__name__:8s} {(time.perf_counter() - t0) / 10 * 1000:6.1f} ms je Durchlauf (T = 1024, 12 Köpfe, Gerät: {q.device})")

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#107C72; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right"><span style="border:1px solid rgba(255,255,255,0.85); color:#fff; font-size:0.72em; font-weight:bold; padding:1px 9px; border-radius:10px; margin-left:8px; vertical-align:1px;">NICHT PRÜFUNGSRELEVANT</span></span>Vertiefung: KV-Cache</div>
<div style="background:#E7F4F2; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Bei der Textgenerierung kommt ein Token nach dem anderen dazu. Um Token t + 1 vorherzusagen, braucht die letzte Position ihre Query und alle Keys und Values bis t. Wegen der Causal Mask sind die Keys und Values der alten Positionen dieselben wie im Schritt davor, also kann man sie speichern, statt sie neu zu rechnen. Diesen Speicher nennt man KV-Cache. Damit zerfällt die Textgenerierung in zwei Phasen, die sich in der Attention unterscheiden:</p>
<table>
<tr><th>Phase</th><th>Queries je Schicht</th><th>Form der Attention-Gewichte</th><th>Wächst mit T</th></tr>
<tr><td>Prompt verarbeiten („Prefill“)</td><td>alle T Positionen</td><td><code>[B, h, T, T]</code></td><td>quadratisch</td></tr>
<tr><td>ein weiteres Token erzeugen, mit Cache („Decode“)</td><td>nur die neue Position</td><td><code>[B, h, 1, T]</code></td><td>Cache linear, Gewichte linear</td></tr>
</table>
<p>Die quadratische Gewichtsmatrix aus Kapitel 3.4 entsteht also nur im Prefill. Beim Erzeugen eines einzelnen Tokens wird kein neues T × T gebildet, dafür wächst der Cache mit jedem Token. Sein Speicher beträgt allgemein 2 · B · L · T · h_KV · d_h · Bytes je Wert, mit L Schichten, h_KV Key/Value-Köpfen und d_h Dimensionen je Kopf. Bei gewöhnlicher Multi-Head Attention ist h_KV · d_h = d, also 2 · d Werte je Token und Schicht. Für GPT-2 small mit 12 Schichten sind das 2 · 768 · 12 = 18.432 Zahlen je Token, in float32 72 KiB. Bei 1.024 Tokens und Batch 8 sind das 576 MiB, mehr als die 475 MiB der Modellparameter, ebenfalls in float32. Bei langen Kontexten und vielen gleichzeitigen Anfragen kann dieser Cache der größte Speicherposten der Inferenz werden. An der Formel sieht man auch, was geteilte Keys und Values bringen: Für eine Sequenz mit T = 1.024 in float32 ist der Cache von GPT-2 small h_KV · 6 MiB, also 72 MiB bei 12 Key/Value-Köpfen, 24 MiB bei Grouped-Query Attention mit vier Gruppen und 6 MiB bei Multi-Query Attention (Aufgabe 3.4 #1). Die Zahl der Query-Köpfe und die Form der Ausgabe ändern sich dabei nicht. Drei Projektthemen setzen hier an: I5 verwaltet den Cache in Blöcken (PagedAttention: Speicher wird erst bei Bedarf blockweise vergeben, freigegebene Blöcke werden wiederverwendet, der einzelne Key- oder Value-Vektor bleibt dabei gleich groß), I8 verkleinert ihn über h_KV, also durch geteilte Keys und Values, und I9 untersucht, was passiert, wenn man alte Einträge verwirft. Eine Implementierung in wenigen Zeilen steht in Raschkas Repository im Bonusmaterial zu Kapitel 4. Die Zelle darunter zeigt den Decode-Schritt an unserer <code>mha_gpt2</code>: dieselbe letzte Ausgabe einmal aus dem vollen Durchlauf, einmal aus gespeicherten Keys und Values.</p>

</div>
</div>

<details style="margin:4px 0 10px"><summary style="cursor:pointer; color:#156082; font-weight:600;">Code Schritt für Schritt</summary><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code><span style="color: #3D7B7B; font-style: italic"># Vorgabe: ein Decode-Schritt mit KV-Cache — einfach ausführen</span>&#10;mha_gpt2<span style="color: #666">.</span>eval()&#10;<span style="color: #008000; font-weight: bold">with</span> torch<span style="color: #666">.</span>no_grad():&#10;    b_, T_, _ <span style="color: #666">=</span> x<span style="color: #666">.</span>shape</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>①</b>&nbsp; Referenz: der volle Durchlauf über alle T Positionen, wie im Prefill. Uns interessiert nur die letzte Position.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    voll <span style="color: #666">=</span> mha_gpt2(x)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>②</b>&nbsp; Der Cache: Keys und Values der ersten T − 1 Positionen liegen aus früheren Schritten schon vor. Hier rechnen wir sie einmal aus, um den Zustand nachzustellen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    <span style="color: #008000; font-weight: bold">def</span><span style="color: #BBB"> </span><span style="color: #00F">koepfe</span>(t):&#10;        <span style="color: #008000; font-weight: bold">return</span> t<span style="color: #666">.</span>view(b_, <span style="color: #666">-1</span>, <span style="color: #666">12</span>, <span style="color: #666">64</span>)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)&#10;    k_cache <span style="color: #666">=</span> koepfe(mha_gpt2<span style="color: #666">.</span>W_key(x[:, :<span style="color: #666">-1</span>]))&#10;    v_cache <span style="color: #666">=</span> koepfe(mha_gpt2<span style="color: #666">.</span>W_value(x[:, :<span style="color: #666">-1</span>]))</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>③</b>&nbsp; Das neue Token: Nur seine Query, sein Key und sein Value werden berechnet, Key und Value kommen hinten an den Cache.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    q_neu <span style="color: #666">=</span> koepfe(mha_gpt2<span style="color: #666">.</span>W_query(x[:, <span style="color: #666">-1</span>:]))&#10;    K <span style="color: #666">=</span> torch<span style="color: #666">.</span>cat([k_cache, koepfe(mha_gpt2<span style="color: #666">.</span>W_key(x[:, <span style="color: #666">-1</span>:]))], dim<span style="color: #666">=2</span>)&#10;    V <span style="color: #666">=</span> torch<span style="color: #666">.</span>cat([v_cache, koepfe(mha_gpt2<span style="color: #666">.</span>W_value(x[:, <span style="color: #666">-1</span>:]))], dim<span style="color: #666">=2</span>)</code></pre><div style="margin:8px 0 2px; color:#475569;"><b>④</b>&nbsp; Attention nur für die eine Query gegen alle T Keys: Gewichte [B, h, 1, T] statt [B, h, T, T]. Eine Maske braucht es nicht, die neue Position darf alle bisherigen sehen.</div><pre style="background:#F6F8FA; border:1px solid #E2E8F0; border-radius:4px; padding:7px 10px; font-size:0.82em; line-height:1.4; overflow-x:auto; margin:2px 0 8px; text-align:left;"><code>    gewichte <span style="color: #666">=</span> torch<span style="color: #666">.</span>softmax(q_neu <span style="color: #666">@</span> K<span style="color: #666">.</span>transpose(<span style="color: #666">2</span>, <span style="color: #666">3</span>) <span style="color: #666">/</span> <span style="color: #666">64**0.5</span>, dim<span style="color: #666">=-1</span>)&#10;    kontext <span style="color: #666">=</span> (gewichte <span style="color: #666">@</span> V)<span style="color: #666">.</span>transpose(<span style="color: #666">1</span>, <span style="color: #666">2</span>)<span style="color: #666">.</span>contiguous()<span style="color: #666">.</span>view(b_, <span style="color: #666">1</span>, <span style="color: #666">768</span>)&#10;    aus_cache <span style="color: #666">=</span> mha_gpt2<span style="color: #666">.</span>out_proj(kontext)&#10;&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">"Gewichte im Decode-Schritt:"</span>, <span style="color: #008000">tuple</span>(gewichte<span style="color: #666">.</span>shape), <span style="color: #BA2121">" im Prefill:"</span>, (b_, <span style="color: #666">12</span>, T_, T_))&#10;    <span style="color: #008000">print</span>(<span style="color: #BA2121">"gleiche letzte Ausgabe:"</span>, torch<span style="color: #666">.</span>allclose(aus_cache, voll[:, <span style="color: #666">-1</span>:], atol<span style="color: #666">=1e-5</span>))</code></pre></details>

In [ ]:
# Vorgabe: ein Decode-Schritt mit KV-Cache — einfach ausführen
mha_gpt2.eval()
with torch.no_grad():
    b_, T_, _ = x.shape
    voll = mha_gpt2(x)

    def koepfe(t):
        return t.view(b_, -1, 12, 64).transpose(1, 2)
    k_cache = koepfe(mha_gpt2.W_key(x[:, :-1]))
    v_cache = koepfe(mha_gpt2.W_value(x[:, :-1]))

    q_neu = koepfe(mha_gpt2.W_query(x[:, -1:]))
    K = torch.cat([k_cache, koepfe(mha_gpt2.W_key(x[:, -1:]))], dim=2)
    V = torch.cat([v_cache, koepfe(mha_gpt2.W_value(x[:, -1:]))], dim=2)

    gewichte = torch.softmax(q_neu @ K.transpose(2, 3) / 64**0.5, dim=-1)
    kontext = (gewichte @ V).transpose(1, 2).contiguous().view(b_, 1, 768)
    aus_cache = mha_gpt2.out_proj(kontext)

    print("Gewichte im Decode-Schritt:", tuple(gewichte.shape), " im Prefill:", (b_, 12, T_, T_))
    print("gleiche letzte Ausgabe:", torch.allclose(aus_cache, voll[:, -1:], atol=1e-5))

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#107C72; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;"><span style="float:right"><span style="border:1px solid rgba(255,255,255,0.85); color:#fff; font-size:0.72em; font-weight:bold; padding:1px 9px; border-radius:10px; margin-left:8px; vertical-align:1px;">NICHT PRÜFUNGSRELEVANT</span></span>Vertiefung: Leseliste: wie aktuelle Modelle die Attention verändern</div>
<div style="background:#E7F4F2; color:#1a1a1a; padding:12px 16px; margin:0;">
<p>Softmax-Attention bleibt in allen genannten Modellen enthalten, in den Hybriden aber nur noch in einem Teil der Schichten. Die Tabelle ordnet die Verfahren nach dem Ort, an dem sie in die Rechnung aus Kapitel 3.4 eingreifen (dasselbe Raster wie die Übersichtsfolie), und nennt je Verfahren, was sich ändert und welches Problem das löst.</p>
<table>
<tr><th>Eingriffsort</th><th>Verfahren</th><th>Was sich ändert</th><th>Nutzen</th></tr>
<tr><td>Speicherverwaltung: Wo liegen die K/V-Einträge?</td><td>KV-Cache, <a href="https://arxiv.org/abs/2309.06180">PagedAttention</a> (Kwon et al. 2023)</td><td>Keys und Values vergangener Positionen werden gespeichert statt neu berechnet, ihr Speicher in Blöcken erst bei Bedarf vergeben.</td><td>Decode-Schritte ohne neues T × T, kaum Verschnitt im Speicher. Projektthema I5.</td></tr>
<tr><td>K/V-Darstellung: Wie viele Zahlen je Position?</td><td><a href="https://arxiv.org/abs/2305.13245">Grouped-Query Attention</a> (Ainslie et al. 2023), Multi-Query Attention (Aufgabe 3.4 #1)</td><td>Mehrere Query-Köpfe teilen sich Key und Value. Die Query-Köpfe und ihre Gewichtsverteilungen bleiben erhalten.</td><td>Kleinerer KV-Cache über h_KV, etwa gpt-oss mit 64 Query- und 8 KV-Köpfen laut <a href="https://arxiv.org/abs/2508.10925">Model Card</a>.</td></tr>
<tr><td>K/V-Darstellung</td><td><a href="https://arxiv.org/abs/2405.04434">Multi-Head Latent Attention</a> (DeepSeek-V2 2024)</td><td>Keys und Values werden aus einem kleinen latenten Vektor je Token rekonstruiert, gespeichert wird nur dieser.</td><td>KV-Cache schrumpft deutlich, die Köpfe bleiben verschieden.</td></tr>
<tr><td>Betrachtete Positionen: Welche Keys sieht eine Query?</td><td>Lokale Fensterschichten (<a href="https://arxiv.org/abs/2503.19786">Gemma 3</a>)</td><td>Die meisten Schichten sehen nur ein Fenster von 1.024 Positionen, jede sechste Schicht den ganzen Kontext.</td><td>KV-Cache und Rechenzeit der lokalen Schichten hängen nicht mehr von T ab.</td></tr>
<tr><td>Rekurrenter Zustand statt wachsendem Cache</td><td>Hybride mit <a href="https://arxiv.org/abs/2412.06464">Gated DeltaNet</a> (Yang et al. 2025)</td><td>Drei von vier Schichten ersetzen die Softmax-Attention durch Gated-DeltaNet-Schichten mit linearer Sequenzkomplexität und einem Zustand fester Größe, so in <a href="https://huggingface.co/Qwen/Qwen3-Next-80B-A3B-Instruct">Qwen3-Next</a>.</td><td>Speicher und Rechenzeit wachsen in diesen Schichten nicht mit T.</td></tr>
<tr><td>Betrachtete Positionen</td><td>Sparse Attention mit Indexer (<a href="https://arxiv.org/abs/2512.02556">DeepSeek-V3.2</a>)</td><td>Ein kleiner Indexer wählt je Query die 2.048 wichtigsten Positionen, nur die gehen in die Attention. Der Indexer selbst bewertet weiter alle Paare.</td><td>Weniger Rechenzeit in der Attention bei langem Kontext.</td></tr>
<tr><td>Betrachtete Positionen (Streaming)</td><td>Attention Sinks (<a href="https://arxiv.org/abs/2309.17453">Xiao et al. 2024</a>)</td><td>Bei festem Cache-Budget bleiben wenige Anfangspositionen dauerhaft im Cache, dazu ein Fenster der jüngsten Tokens. Hintergrund: Viele Köpfe legen hohes Gewicht auf die ersten Positionen, unabhängig von deren Inhalt.</td><td>Generierung über das Budget hinaus ohne Qualitätseinbruch. Projektthema I9.</td></tr>
<tr><td>Ausgabesteuerung: Wie viel trägt die Mischung bei?</td><td>Gelernter Sink (gpt-oss, <a href="https://arxiv.org/abs/2508.10925">Model Card</a>)</td><td>Je Kopf ein zusätzlicher gelernter Logit s in der Softmax, der zu keinem Token gehört: a<sub>j</sub> = exp(z<sub>j</sub>) / (exp(s) + Σ<sub>i</sub> exp(z<sub>i</sub>)). Die Gewichte über die echten Tokens summieren sich dann nicht mehr zwingend zu 1, ein Kopf kann „nirgends“ hinschauen.</td><td>Stabilität. Anders als die erhaltenen Anfangstokens beim Streaming kostet der Sink keinen Cache-Platz.</td></tr>
<tr><td>Ausgabesteuerung</td><td>Gate nach der Attention (<a href="https://arxiv.org/abs/2505.06708">Qiu et al. 2025</a>)</td><td>Die Mischung wird vor der Ausgabeprojektion elementweise mit einem gelernten Gate skaliert.</td><td>Stabileres Training, weniger Attention Sinks.</td></tr>
<tr><td>Ausführung auf der Hardware</td><td><a href="https://arxiv.org/abs/2205.14135">FlashAttention</a> (Dao et al. 2022)</td><td>Dieselbe dichte Attention, blockweise im schnellen Speicher des Chips gerechnet, die Gewichtsmatrix wird nie vollständig abgelegt (Vertiefung oben).</td><td>Weniger Speichertransfers, schneller. Der Rechenaufwand bleibt quadratisch.</td></tr>
</table>
<p>Die Folien zeigen dazu eine Übersicht und je Verfahren eine eigene Folie.</p>
<p><strong>Zum Ansehen im Browser.</strong> <a href="https://poloclub.github.io/transformer-explainer/">Transformer Explainer</a> lässt GPT-2 small live im Browser laufen und zeigt jede Attention-Operation. <a href="https://bbycroft.net/llm">LLM Visualization</a> zeigt ein GPT-Modell in 3D. Beide laufen ohne Installation. <a href="https://github.com/jessevig/bertviz">BertViz</a> braucht Python, das README verlinkt ein fertiges Colab-Notebook mit geladenem Modell.</p>

</div>
</div>

<div style="border:2px solid #7E3FA0; background:#fff; color:#1a1a1a; padding:12px 16px; border-radius:6px; ">
<div style="overflow:hidden; margin-bottom:4px;"><span style="color:#7E3FA0; font-weight:bold; font-size:1.08em;">Lernfragen 3.5 #1</span></div>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">1.</b> Beim Decode-Schritt werden Keys und Values der alten Positionen aus dem Cache geholt. Müssen auch ihre Queries gespeichert werden?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">2.</b> Wird ein gespeicherter Key- oder Value-Vektor durch PagedAttention kürzer?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">3.</b> Zwei Köpfe teilen sich bei Multi-Query Attention dieselben Keys und Values. Erzeugen sie zwangsläufig dieselbe Ausgabe?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">4.</b> Grouped-Query Attention mit vier Gruppen verkleinert den KV-Cache von GPT-2 small auf ein Drittel. Ist die Inferenz damit dreimal so schnell?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">5.</b> StreamingLLM behält die ersten Positionen und ein Fenster der jüngsten Tokens. Kann das Modell Information aus der verworfenen Mitte später gezielt abrufen?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">6.</b> Ist der gelernte Sink in gpt-oss ein erhaltenes Wort vom Anfang des Prompts?</p>
<p style="padding-left:1.2em; text-indent:-1.2em;"><b style="color:#7E3FA0;">7.</b> Ist FlashAttention eine Näherung, die nur einen Teil der Tokens betrachtet?</p>

</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">➡️ Nächste Schritte</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<ul>
<li>Pflicht bis zum nächsten Termin: 3.1 #1 (Handrechnung) und 3.2 #2 (Skalierung), zusammen rund 20 Minuten, dazu die Vorlesungsaufgaben, die Sie im Termin nicht abgeschlossen haben (3.3 #1, 3.4 #1, 3.4 #3). Die mit „nach Wahl“ markierten Aufgaben sind freiwillig.</li>
<li>Schlusskontrolle: Starten Sie den Kernel neu und führen Sie das Notebook von oben aus. Bei einer Aufgabe, die Sie nicht bearbeitet haben, überspringen Sie die Aufgabenzelle und ihre Prüfzelle. Alle anderen Zellen müssen durchlaufen.</li>
<li>Lesen Sie Raschka, Kapitel 4 bis einschließlich Abschnitt 4.3.</li>
<li>Zum Nachlesen: Jay Alammar, *The Illustrated Transformer*. 3Blue1Brown, *Attention in transformers, step-by-step*.</li>
</ul>
<p><strong>→ Ausblick:</strong> <code>MultiHeadAttention</code> ist fertig. In der nächsten Lerneinheit kommen Residualverbindung, LayerNorm und Feed-Forward-Netz hinzu. Zusammen mit der Attention bilden sie den Transformer-Block, und zwölf solcher Blöcke ergeben GPT-2 small.</p>

</div>
</div>

<div style="border-radius:6px; overflow:hidden; margin:2px 0; margin-right:-20px;">
<div style="background:#E8B029; color:#fff; padding:6px 16px; font-weight:bold; font-size:1.08em; margin:0; overflow:hidden;">Quellen und Referenzen</div>
<div style="background:#FDF6E3; color:#1a1a1a; padding:12px 16px; margin:0;">
<ul>
<li>Sebastian Raschka: *Build a Large Language Model (From Scratch)*, Manning 2024, Kapitel 3. Code: <a href="https://github.com/rasbt/LLMs-from-scratch/tree/main/ch03">rasbt/LLMs-from-scratch</a>, Lizenz Apache 2.0. Die Klassen <code>SelfAttention_v1</code>, <code>SelfAttention_v2</code>, <code>CausalAttention</code> und <code>MultiHeadAttention</code> sind daraus übernommen und angepasst (<code>return_weights</code>, deutsche Kommentare).</li>
<li>Ashish Vaswani et al.: *Attention Is All You Need*, NeurIPS 2017, arXiv:1706.03762. Skalierung (Abschnitt 3.2.1), Multi-Head (3.2.2), Maske (3.2.3).</li>
<li>Dzmitry Bahdanau, Kyunghyun Cho, Yoshua Bengio: *Neural Machine Translation by Jointly Learning to Align and Translate*, ICLR 2015, arXiv:1409.0473. Der Ursprung der gewichteten Summe.</li>
<li>Noam Shazeer: *Fast Transformer Decoding: One Write-Head is All You Need*, arXiv:1911.02150. Multi-Query Attention, Aufgabe 3.4 #1.</li>
<li>Tri Dao et al.: *FlashAttention*, NeurIPS 2022, arXiv:2205.14135. Hintergrund der ersten Vertiefung.</li>
<li>Sarthak Jain, Byron C. Wallace: *Attention is not Explanation*, NAACL 2019. Sarah Wiegreffe, Yuval Pinter: *Attention is not not Explanation*, EMNLP 2019. Hintergrund der Denkfrage 3.4 #5.</li>
<li>Leseliste: Olsson et al. 2022, arXiv:2209.11895 · Wang et al. 2022, arXiv:2211.00593 · McDougall et al. 2023, arXiv:2310.04625 · Yin, Steinhardt 2025, ICML, arXiv:2502.14010 · Ainslie et al. 2023, arXiv:2305.13245 · DeepSeek-AI 2024, arXiv:2405.04434 · Gemma Team 2025, arXiv:2503.19786 · Yang et al. 2025, arXiv:2412.06464 · DeepSeek-AI 2025, arXiv:2512.02556 · OpenAI 2025, gpt-oss Model Card, arXiv:2508.10925 · Qiu et al. 2025, arXiv:2505.06708 · Xiao et al. 2024, arXiv:2309.17453 · Kwon et al. 2023, arXiv:2309.06180 · Dao et al. 2022, arXiv:2205.14135.</li>
<li>Beispieltext: Edith Wharton, *The Verdict* (1908), gemeinfrei.</li>
</ul>

</div>
</div>